# 29 · School access dashboard — Part D, step D8 (schools)

One **self-contained HTML file** (`FINAL COMPLETE DATASET/results/dashboard/school_access_dashboard.html`): open it
in any browser, no internet needed (Leaflet 1.9.4 is inlined; no background map, so it is fully offline).

- **India:** states and districts with every level and measure (% of children beyond the distance norm, mean distance
  to the nearest school, seats per child norm-based / enrolment);
  sub-districts coloured by area type (K-Means) with Gi* poor-access clusters outlined.
- **Kerala:** districts, taluks and local bodies (Sametham levels LP / UP / HS / HSS), Gi* clusters, and the
  **priority areas for new schools** (clusters of children with no school within the norm; replaces site selection).
- **Panels:** Gini and Theil, area-type profiles, FP-Growth rules and data notes.
- **Click an area:** a plain-language summary of the findings for every school level (children, distance, seats per child).
- State-levels without reliable school data are grey ("no data").

**Checks:** every table row has a shape; the embedded data parses; 10 random units per layer equal the CSVs.

## Project folder (laptop or Colab)

In [1]:
# Project folder. Laptop: open the notebook from "PROJECTS IPYNB" (or the project root).
# Google Colab: upload the project folder to Google Drive ("My Drive") and set its name below.
from pathlib import Path
import sys
PROJECT_FOLDER = "Distance-Capacity-and-Inequality-in-Access-to-Government-and-Aided-Schools"  # Colab only: folder name inside My Drive
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
else:
    PROJECT = Path.cwd().parent if Path.cwd().name in ("notebooks", "PROJECTS IPYNB") else Path.cwd()
assert (PROJECT / "scripts").is_dir(), f"project folder not found: {PROJECT}"
sys.path.insert(0, str(PROJECT / "scripts"))

In [2]:
# Colab only: install packages that Colab does not have by default (does nothing on the laptop)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "pymupdf", "pyogrio", "rasterio"], check=True)

## Settings and Leaflet

In [3]:
import json
import random
import urllib.request

import duckdb
import numpy as np
import pandas as pd
import shapely

sys.path.insert(0, str(PROJECT / "scripts" / "bigdata"))
import population_cells as pc

SILVER, GOLD = PROJECT / "data" / "lake" / "silver", PROJECT / "data" / "lake" / "gold"
OUT = PROJECT / "FINAL COMPLETE DATASET" / "results" / "dashboard"
OUT.mkdir(parents=True, exist_ok=True)
LEVELS = ["primary", "upper_primary", "secondary", "higher_secondary"]
LEVELS_K = ["lp", "up", "hs", "hss"]

# Leaflet 1.9.4 (BSD-2-Clause), inlined so the dashboard works without internet
LEAF = PROJECT / "tools" / "leaflet"
LEAF.mkdir(parents=True, exist_ok=True)
for f in ["leaflet.js", "leaflet.min.css"]:
    if not (LEAF / f).exists():
        req = urllib.request.Request("https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/" + f,
                                     headers={"User-Agent": "msc-spatial-inequality-study/0.1 (academic project)"})
        (LEAF / f).write_bytes(urllib.request.urlopen(req, timeout=60).read())
LEAFLET_JS, LEAFLET_CSS = (LEAF / "leaflet.js").read_text(encoding="utf-8"), (LEAF / "leaflet.min.css").read_text(encoding="utf-8")
assert "1.9.4" in LEAFLET_JS[:300] and len(LEAFLET_JS) > 100_000, "Leaflet download looks wrong"
print("Leaflet", len(LEAFLET_JS) // 1000, "kB inlined")

Leaflet 147 kB inlined


## Read the D6 / D7 results; Kerala taluks

In [4]:
# Results of notebooks 27 (D6) and 28 (D7)
rd = lambda name: pd.read_csv(GOLD / name)
cov = rd("school_data_coverage.csv")
by_state, by_district = rd("school_india_by_state.csv"), rd("school_india_by_district.csv")
by_sub = rd("school_india_by_subdistrict.csv").query("subdistrict_code > 0")
k_district = rd("school_kerala_by_district.csv").query("state == 'Kerala'")
k_lb = rd("school_kerala_by_local_body.csv").query("state == 'Kerala' and local_body_code > 0")
hot_in, hot_k = rd("school_hotspots_india_subdistrict.csv"), rd("school_hotspots_kerala_local_body.csv")
types, profiles = rd("school_area_types_india_subdistrict.csv"), rd("school_area_type_profiles.csv")
rules, clusters = rd("school_fp_rules.csv"), rd("school_underserved_clusters_kerala.csv")
ineq_in, ineq_k = rd("school_india_inequality.csv"), rd("school_kerala_inequality.csv")
FLAG = {lv: set(cov.loc[cov["udise_incomplete"] & cov["level"].eq(lv), "state"]) for lv in LEVELS}

# Area-type names from each K-Means profile (cluster numbers are arbitrary, so names follow the profile)
p = profiles.set_index("cluster")
b = p[[f"beyond_norm_%_{lv}" for lv in LEVELS]]
names, left = {}, list(p.index)
names[b.loc[left].mean(axis=1).idxmax()] = "Remote: far at every level"
left = [c for c in left if c not in names]
names[p.loc[left, "beyond_norm_%_primary"].idxmax()] = "Far from primary school"
left = [c for c in left if c not in names]
names[p.loc[left, "beyond_norm_%_higher_secondary"].idxmax()] = "Higher secondary far"
left = [c for c in left if c not in names]
names[p.loc[left, "a_main_primary"].idxmin()] = "Crowded: near but few places"
left = [c for c in left if c not in names]
assert len(left) == 1 and len(p) == 5
names[left[0]] = "Well served"
types["type"] = types["cluster"].map(names)
print("area types:", {int(k): v for k, v in names.items()})

# Kerala taluks (sub-districts) from the Kerala run, so the Kerala view uses one set of levels (LP / UP / HS / HSS)
agg = []
for lv in LEVELS_K:
    agg += [f"sum(dem_{lv}) AS children_{lv}",
            f"100 * sum(CASE WHEN schools_in_reach_{lv} IS NULL THEN dem_{lv} ELSE 0 END) / sum(dem_{lv}) AS \"beyond_norm_%_{lv}\"",
            f"sum(dem_{lv} * nearest_m_{lv}) / sum(dem_{lv}) / 1000 AS mean_nearest_km_{lv}",
            f"sum(dem_{lv} * coalesce(a_main_{lv}, 0)) / sum(dem_{lv}) AS a_main_{lv}",
            f"sum(dem_{lv} * coalesce(a_enrol_{lv}, 0)) / sum(dem_{lv}) AS a_enrol_{lv}"]
acc_k = (GOLD / "school_access_kerala").as_posix()
cells = (SILVER / "population_cells").as_posix()
fixf = (SILVER / "cell_units_fixed").as_posix()
k_taluk = duckdb.sql(f"""
    WITH c AS (SELECT p.cell_id, coalesce(f.subdistrict_code, p.subdistrict_code) AS subdistrict_code
               FROM read_parquet('{cells}/**/*.parquet', hive_partitioning = true) p
               LEFT JOIN read_parquet('{fixf}') f USING (cell_id)
               WHERE p.state IN ('Kerala', 'Puducherry'))
    SELECT c.subdistrict_code, count(*) AS cells, sum(a.pop) AS pop, {", ".join(agg)}
    FROM read_parquet('{acc_k}/*.parquet') a JOIN c USING (cell_id)
    WHERE a.state = 'Kerala' GROUP BY 1""").df()
tot = duckdb.sql(f"SELECT count(*) n, {', '.join(f'sum(dem_{lv}) AS {lv}' for lv in LEVELS_K)} FROM read_parquet('{acc_k}/*.parquet') WHERE state = 'Kerala'").df().iloc[0]
assert int(k_taluk["cells"].sum()) == int(tot["n"]), "a Kerala cell did not find its taluk"
for lv in LEVELS_K:
    assert abs(k_taluk[f"children_{lv}"].sum() / tot[lv] - 1) < 1e-9
k_taluk = k_taluk[k_taluk["subdistrict_code"] > 0].merge(by_sub[["subdistrict_code", "district", "subdistrict"]], on="subdistrict_code", how="left")
print("Kerala taluks:", len(k_taluk), "| India sub-districts:", len(by_sub), "| districts:", len(by_district), "| states:", len(by_state))

area types: {1: 'Remote: far at every level', 3: 'Far from primary school', 4: 'Higher secondary far', 2: 'Crowded: near but few places', 0: 'Well served'}


Kerala taluks: 77 | India sub-districts: 6810 | districts: 735 | states: 36


## Map layers

In [5]:
# Map layers: simplified shapes (coordinates rounded to 4 decimals, about 10 m) + the values shown in the map and pop-ups
def shapes(code_col, tol):
    g = pc.load_units(SILVER, code_col).set_crs(4326, allow_override=True)
    g["geometry"] = shapely.transform(g.geometry.simplify(tol, preserve_topology=True), lambda c: np.round(c, 4))
    return g[["code", "unit_name", "geometry"]]


def r(v, nd):
    return None if pd.isna(v) else round(float(v), nd)


def values(row, levels):
    out = {}
    for lv in levels:
        out[f"c_{lv}"] = r(row[f"children_{lv}"], 0)
        out[f"b_{lv}"] = r(row[f"beyond_norm_%_{lv}"], 2)  # 2 decimals so national sums match the tables
        out[f"n_{lv}"] = r(row[f"mean_nearest_km_{lv}"], 3)
        out[f"m_{lv}"] = r(row[f"a_main_{lv}"], 3)
        out[f"e_{lv}"] = r(row[f"a_enrol_{lv}"], 3)
    return out


def layer(g, g_key, table, t_key, props):
    """GeoJSON FeatureCollection: every table row must find its shape."""
    m = g.merge(table, left_on=g_key, right_on=t_key, how="inner", suffixes=("_shape", ""))
    assert len(m) == len(table), f"{t_key}: {len(table) - len(m)} rows without a shape"
    feats = [{"type": "Feature", "geometry": shapely.geometry.mapping(row["geometry"]), "properties": props(row)}
             for _, row in m.iterrows()]
    return {"type": "FeatureCollection", "features": feats}


def hot_flags(h, key):
    """Per unit and level: in a significant poor-access cluster (hot spot of far-away children / cold spot of places)."""
    far = h[h["variable"].eq("beyond_norm_%")].pivot(index=key, columns="level", values="poor_access_cluster").fillna(False)
    few = h[h["variable"].eq("a_main")].pivot(index=key, columns="level", values="poor_access_cluster").fillna(False)
    return far.astype(bool), few.astype(bool)


hfar, hfew = hot_flags(hot_in, "subdistrict_code")
kfar, kfew = hot_flags(hot_k, "local_body_code")
type_of = types.set_index("subdistrict_code")["type"]
sub = by_sub[["district_code", "subdistrict_code"]].copy()
for lv in LEVELS:
    sub[lv] = sub["subdistrict_code"].map(hfar[lv] | hfew[lv]).fillna(False).astype(bool)
sub["type"] = sub["subdistrict_code"].map(type_of).fillna("Not classified")
d_types = sub.groupby("district_code")["type"].value_counts().unstack(fill_value=0)


def flags(state):
    return {f"f_{lv}": int(state in FLAG[lv]) for lv in LEVELS}


def hot_props(code, far, few, levels):
    return {**{f"hf_{lv}": int(bool(far[lv].get(code, False))) if lv in far else 0 for lv in levels},
            **{f"hp_{lv}": int(bool(few[lv].get(code, False))) if lv in few else 0 for lv in levels}}


LAYERS = {
    "in_state": layer(shapes("state_code", 0.02), "unit_name", by_state, "state",
                      lambda x: {"name": x["state"], **values(x, LEVELS), **flags(x["state"])}),
    "in_district": layer(shapes("district_code", 0.01), "code", by_district, "district_code",
                         lambda x: {"name": x["district"], "state": x["state"], **values(x, LEVELS), **flags(x["state"]),
                                    "types": {k: int(v) for k, v in d_types.loc[x["district_code"]].items() if v} if x["district_code"] in d_types.index else {}}),
    "in_sub": layer(shapes("subdistrict_code", 0.01), "code", by_sub, "subdistrict_code",
                    lambda x: {"name": x["subdistrict"], "district": x["district"], "state": x["state"],
                               "type": type_of.get(x["subdistrict_code"], "Not classified"), **values(x, LEVELS), **flags(x["state"]),
                               **hot_props(x["subdistrict_code"], hfar, hfew, LEVELS)}),
    "k_district": layer(shapes("district_code", 0.002), "code", k_district, "district_code",
                        lambda x: {"name": x["district"], **values(x, LEVELS_K)}),
    "k_taluk": layer(shapes("subdistrict_code", 0.001), "code", k_taluk, "subdistrict_code",
                     lambda x: {"name": x["subdistrict"], "district": x["district"], "type": type_of.get(x["subdistrict_code"], "Not classified"),
                                **values(x, LEVELS_K)}),
    "k_lb": layer(shapes("local_body_code", 0.0005), "code", k_lb, "local_body_code",
                  lambda x: {"name": x["local_body"], "district": x["district"], **values(x, LEVELS_K),
                             **hot_props(x["local_body_code"], kfar, kfew, LEVELS_K)}),
}
pts = [{"level": c.level, "children": round(c.children, 2), "cells": int(c.cells), "lon": round(c.lon, 4), "lat": round(c.lat, 4),
        "district": c.district, "local_body": c.local_body} for c in clusters.itertuples()]
print({k: len(v["features"]) for k, v in LAYERS.items()}, "| priority areas:", len(pts))

{'in_state': 36, 'in_district': 735, 'in_sub': 6810, 'k_district': 14, 'k_taluk': 77, 'k_lb': 1027} | priority areas: 411


## Holes in the Kerala local-body map (no OpenStreetMap boundary)

In [6]:
# Holes in the Kerala local-body map: OpenStreetMap has 1,027 of Kerala's 1,034 local bodies. Holes over 5 km2 are
# drawn grey with the real numbers of the cells inside them that have no local body (nothing is invented).
import geopandas as gpd
from rapidfuzz import fuzz

lbs = pc.load_units(SILVER, "local_body_code").set_crs(4326, allow_override=True)
kerala = pc.load_units(SILVER, "state_code").set_crs(4326, allow_override=True).query("unit_name == 'Kerala'")
hole = kerala.to_crs(7755).geometry.union_all().difference(lbs.to_crs(7755).geometry.union_all())
holes = gpd.GeoDataFrame(geometry=[hole], crs=7755).explode(index_parts=False).reset_index(drop=True)
holes = holes[holes.area > 5e6].reset_index(drop=True)  # 5 km2 leaves out thin slivers along the coast and borders
holes["km2"] = holes.area / 1e6
holes = holes.to_crs(4326)
where = gpd.GeoDataFrame(geometry=holes.representative_point(), crs=4326)
for col in ["district_code", "subdistrict_code"]:
    u = pc.load_units(SILVER, col).set_crs(4326, allow_override=True)[["unit_name", "geometry"]]
    holes[col.split("_")[0]] = gpd.sjoin(where, u, predicate="within")["unit_name"].groupby(level=0).first().reindex(where.index).to_numpy()
# local bodies of each hole's area that are not in OpenStreetMap (checked by name below)
MISSING = {"Chirayinkeezhu": ["Kilimanoor", "Pazhayakunnummel", "Nagaroor", "Pulimath"],
           "Hosdurg": ["Kanhangad", "Nileshwaram"], "Thalassery": ["Chittariparamba"]}
assert sorted(holes["subdistrict"]) == sorted(MISSING), holes[["subdistrict", "km2"]]
assert len(sum(MISSING.values(), [])) == 1034 - len(lbs), "the missing names must explain all 1,034 - 1,027 local bodies"
osm_names = [n.lower() for n in lbs["unit_name"]]
for nm in sum(MISSING.values(), []):
    close = [n for n in osm_names if nm.lower() in n or fuzz.ratio(nm.lower(), n) >= 85]
    assert not close, (nm, close)

k0 = duckdb.sql(f"SELECT * FROM read_parquet('{acc_k}/*.parquet') WHERE state = 'Kerala' AND local_body_code = 0").df()
pts0 = gpd.GeoDataFrame(k0[["cell_id"]], geometry=gpd.points_from_xy(k0["lon"], k0["lat"]), crs=4326)
inh = gpd.sjoin(pts0, holes[["geometry"]], predicate="within")[["cell_id", "index_right"]].rename(columns={"index_right": "hole"})
k0h = k0.merge(inh, on="cell_id")
hv = duckdb.sql(f"SELECT hole, count(*) AS cells, sum(pop) AS pop, {', '.join(agg)} FROM k0h GROUP BY 1").df().set_index("hole")
assert hv.index.is_unique and len(hv) == len(holes) and (hv["cells"] > 0).all(), "a hole has no cells"
holes["geometry"] = shapely.transform(holes.geometry.simplify(0.0005, preserve_topology=True), lambda c: np.round(c, 4))
GAPS = {"type": "FeatureCollection", "features": [
    {"type": "Feature", "geometry": shapely.geometry.mapping(h.geometry),
     "properties": {"name": f"No local-body boundary in OpenStreetMap ({h.subdistrict} taluk)", "district": h.district,
                    "note": f"Local bodies here not in OpenStreetMap: {', '.join(MISSING[h.subdistrict])}. "
                            f"{h.km2:.0f} km²; values = the {int(hv.loc[i, 'cells']):,} cells of 100 m inside with no local body "
                            f"({hv.loc[i, 'pop']:,.0f} people); cells within about 0.5 km of a mapped neighbour were given "
                            f"that neighbour in step D2. Everyone here is counted in the district and taluk maps.",
                    **values(hv.loc[i], LEVELS_K)}} for i, h in holes.iterrows()]}
print(holes.assign(cells=hv["cells"], pop=hv["pop"].round(), children_lp=hv["children_lp"].round(),
                   beyond_lp=hv["beyond_norm_%_lp"].round(1)).drop(columns="geometry").round(1).to_string())
print(f"cells with no local body: {len(k0):,} ({k0['pop'].sum():,.0f} people); inside the {len(holes)} holes: {len(k0h):,}")

    km2            district     subdistrict  cells       pop  children_lp  beyond_lp
0  66.4           Kasaragod         Hosdurg   5635  128101.0       3396.0       16.1
1  34.9              Kannur      Thalassery   1768   10824.0        287.0       27.4
2  98.0  Thiruvananthapuram  Chirayinkeezhu   8110   86146.0       2284.0       23.0
cells with no local body: 15,609 (225,633 people); inside the 3 holes: 15,513


## Side panels

In [7]:
# Side panels (HTML tables built here from the CSVs)
def table(df, fmt=None):
    return df.to_html(index=False, border=0, classes="t", float_format=fmt or (lambda v: f"{v:,.3f}"), na_rep="–")


LV = {"primary": "Primary (1–5)", "upper_primary": "Upper primary (6–8)", "secondary": "Secondary (9–10)",
      "higher_secondary": "Higher sec. (11–12)", "lp": "LP (1–4)", "up": "UP (5–7)", "hs": "HS (8–10)", "hss": "HSS (+1/+2)"}
VAR = {"a_main": "seats per child (norm)", "a_enrol": "seats per child (enrolment)", "near": "distance to nearest school"}
gi = ineq_in[ineq_in["scope"].eq("India")].copy()
gi["level"], gi["variable"] = gi["level"].map(LV), gi["variable"].map(VAR)
gi = gi.rename(columns={"between_share_%": "between states %"})[["scope", "level", "variable", "gini", "theil", "between states %"]]
gk = ineq_k.assign(level=ineq_k["level"].map(LV), variable=ineq_k["variable"].map(VAR))[
    ["level", "variable", "gini", "theil", "between_district_%", "between_local_body_%"]].rename(
    columns={"between_district_%": "between districts %", "between_local_body_%": "between local bodies %"})
prof = profiles.assign(type=profiles["cluster"].map(names))
pt = pd.DataFrame({"area type": prof["type"], "sub-districts": prof["subdistricts"], "top states": prof["top_states"],
                   **{f"{LV[lv]}: % beyond norm": prof[f"beyond_norm_%_{lv}"] for lv in LEVELS},
                   **{f"{LV[lv]}: seats / child": prof[f"a_main_{lv}"] for lv in LEVELS}})
lab = lambda s: " + ".join(i.replace("few_places_", "few seats: ").replace("far_", "far: ").replace("_", " ") for i in s.split(" + "))
rt = rules.head(10).assign(antecedent=lambda d: d["antecedent"].map(lab), consequent=lambda d: d["consequent"].map(lab))[
    ["antecedent", "consequent", "support", "confidence", "lift"]].rename(columns={"antecedent": "if", "consequent": "then"})
PANELS = {
    "Inequality": "<h3>India: Gini and Theil (children-weighted, 140 M cells)</h3>" + table(gi)
                  + "<p class=n>Between states % = share of Theil inequality between states (the rest is within states).</p>"
                  + "<h3>Kerala (Sametham levels)</h3>" + table(gk),
    "Area types": "<h3>Sub-district area types (K-Means, K = 5, silhouette 0.42)</h3>" + table(pt, lambda v: f"{v:,.2f}")
                  + "<p class=n>Children-weighted means. Grey on the map = not classified (too few children or no data).</p>",
    "Rules": "<h3>Problems that occur together (FP-Growth, 6,810 sub-districts)</h3>" + table(rt)
             + "<p class=n>far = in the worst quarter for % of children beyond the norm; few seats = lowest quarter of seats per child. "
               "Lift = how many times more often than by chance.</p>",
    "Notes": "<h3>Data and methods</h3><ul>"
             "<li>Children: WorldPop 2026 100 m (140 M cells), ages per level × Government + Aided share (UDISE 2022).</li>"
             "<li>Schools: UDISE 2022 Government + Aided (India); Sametham 2026-27 + HSCAP/VHSCAP (Kerala).</li>"
             "<li>Norms: primary/LP 1 km, upper primary/UP 3 km (RTE Rules), secondary/HS 5 km (3 km hilly), higher secondary 7 km (Samagra Shiksha).</li>"
             "<li>Seats per child: E2SFCA with Gaussian distance decay (σ = norm); norm-based seats (teachers × pupil–teacher norm), sensitivity = enrolment.</li>"
             "<li>Priority areas (Kerala): connected 100 m cells with no school of the level within the norm, ≥ 50 children.</li>"
             "<li>Computed with Apache Spark 3.5 + Sedona 1.9 (notebooks 24–28).</li>"
             "<li>Kerala local bodies: OpenStreetMap has 1,027 of 1,034. The holes over 5 km² are grey with a dashed edge ("
             + "; ".join(f"{', '.join(v)} ({k} taluk)" for k, v in MISSING.items())
             + "); their people are counted in the district and taluk maps.</li></ul>"
             "<p class=n>Exact sites for new schools were not optimised; they would also need land, cost and local data.</p>",
}

## Write the HTML

In [8]:
# The page: Leaflet map + controls + panels, everything in one file
TEMPLATE = """<!doctype html><html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1"><title>School Access Explorer</title>
<script>try{var t=localStorage.getItem("sae-theme");var h=(location.hash.match(/theme=(light|dark)/)||[])[1];if(h)t=h;if(t==="light"||t==="dark")document.documentElement.setAttribute("data-theme",t);}catch(e){}</script>
<style>__LEAFLET_CSS__</style><style>
/* ---------- design tokens ---------- */
:root{--s1:4px;--s2:8px;--s3:12px;--s4:16px;--s5:24px;--s6:32px;
 --t-xs:11.5px;--t-sm:12.5px;--t-md:14px;--t-lg:16px;--t-xl:20px;--t-2xl:28px;--t-3xl:34px;
 --r-sm:8px;--r-md:12px;--r-lg:16px;--font:system-ui,-apple-system,"Segoe UI",Roboto,sans-serif;
 --dur:180ms;--ease:cubic-bezier(.2,.7,.2,1)}
:root{color-scheme:light;--page:#f2f3f1;--panel:#f8f8f6;--card:#ffffff;--card2:#f4f4f1;--border:rgba(11,11,11,.09);--border2:rgba(11,11,11,.15);
 --ink:#0b0b0b;--ink2:#47463f;--muted:#66655f;--acc:#2a78d6;--acc-ink:#1c5cab;--accsoft:#e6f0fc;--warn:#b3400b;--track:#e9e8e3;
 --mapbg:#e9ecef;--edge:#ffffff;--stateline:rgba(11,11,11,.45);--nodata:#b4b3ac;
 --e1:0 1px 2px rgba(16,18,20,.06);--e2:0 1px 2px rgba(16,18,20,.05),0 8px 24px rgba(16,18,20,.07);--focus:0 0 0 3px rgba(42,120,214,.35)}
@media (prefers-color-scheme:dark){:root:where(:not([data-theme=light])){color-scheme:dark;--page:#0b0b0c;--panel:#121213;--card:#1a1a1b;--card2:#222221;--border:rgba(255,255,255,.08);--border2:rgba(255,255,255,.14);
 --ink:#f0efec;--ink2:#c9c8bf;--muted:#a3a198;--acc:#4a92ea;--acc-ink:#8cbcf3;--accsoft:#13263b;--warn:#f3925f;--track:#2b2b29;
 --mapbg:#1d2126;--edge:rgba(255,255,255,.55);--stateline:rgba(255,255,255,.55);--nodata:#8d8c86;
 --e1:0 1px 2px rgba(0,0,0,.5);--e2:0 1px 2px rgba(0,0,0,.45),0 10px 28px rgba(0,0,0,.45);--focus:0 0 0 3px rgba(74,146,234,.45)}}
:root[data-theme=dark]{color-scheme:dark;--page:#0b0b0c;--panel:#121213;--card:#1a1a1b;--card2:#222221;--border:rgba(255,255,255,.08);--border2:rgba(255,255,255,.14);
 --ink:#f0efec;--ink2:#c9c8bf;--muted:#a3a198;--acc:#4a92ea;--acc-ink:#8cbcf3;--accsoft:#13263b;--warn:#f3925f;--track:#2b2b29;
 --mapbg:#1d2126;--edge:rgba(255,255,255,.55);--stateline:rgba(255,255,255,.55);--nodata:#8d8c86;
 --e1:0 1px 2px rgba(0,0,0,.5);--e2:0 1px 2px rgba(0,0,0,.45),0 10px 28px rgba(0,0,0,.45);--focus:0 0 0 3px rgba(74,146,234,.45)}
/* ---------- base ---------- */
*{box-sizing:border-box}html,body{height:100%}html{scroll-behavior:smooth}
body{margin:0;background:var(--page);color:var(--ink);font:var(--t-md)/1.45 var(--font);-webkit-font-smoothing:antialiased}
button,select,input{font:inherit;color:inherit}
:focus-visible{outline:none;box-shadow:var(--focus);border-radius:var(--r-sm)}
.sr{position:absolute;width:1px;height:1px;overflow:hidden;clip:rect(0 0 0 0);white-space:nowrap}
/* ---------- header ---------- */
header{position:sticky;top:0;height:60px;display:flex;align-items:center;gap:var(--s5);padding:0 var(--s5);background:var(--panel);border-bottom:1px solid var(--border);z-index:1000}
.brand{display:flex;align-items:center;gap:var(--s3);min-width:0}
.logo{width:32px;height:32px;border-radius:9px;background:linear-gradient(140deg,#3987e5,#184f95);display:grid;place-items:center;flex:none;box-shadow:var(--e1)}
.brand h1{font-size:var(--t-lg);margin:0;line-height:1.15;white-space:nowrap;letter-spacing:-.01em}
.brand p{margin:2px 0 0;color:var(--muted);font-size:var(--t-xs);white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
nav.pages{position:relative;display:flex;gap:2px;margin-left:auto;height:100%}
nav.pages button{position:relative;border:0;background:none;padding:0 var(--s4);color:var(--ink2);font-weight:600;font-size:13.5px;cursor:pointer;height:100%;transition:color var(--dur) var(--ease)}
nav.pages button:hover{color:var(--ink)}nav.pages button.on{color:var(--acc-ink)}
nav.pages .ind{position:absolute;bottom:-1px;height:2px;background:var(--acc);border-radius:2px;transition:left var(--dur) var(--ease),width var(--dur) var(--ease)}
.iconbtn{width:36px;height:36px;display:grid;place-items:center;border:1px solid var(--border);border-radius:10px;background:var(--card);color:var(--ink2);cursor:pointer;transition:background var(--dur),color var(--dur)}
.iconbtn:hover{color:var(--ink);background:var(--card2)}
/* ---------- views ---------- */
.view{display:none}.view.on{display:flex}.page.view.on{display:block}
#app{height:calc(100dvh - var(--hh,60px))}
#side{width:400px;flex:none;overflow-y:auto;background:var(--panel);padding:var(--s4);display:flex;flex-direction:column;gap:var(--s3);border-right:1px solid var(--border)}
#main{flex:1;display:flex;flex-direction:column;min-width:0;background:var(--page)}
#mapwrap{position:relative;flex:1;min-height:0;z-index:0}
#map{position:absolute;inset:0;background:var(--mapbg)}
/* ---------- cards & sections ---------- */
.card{background:var(--card);border:1px solid var(--border);border-radius:var(--r-lg);padding:var(--s4);box-shadow:var(--e1)}
.sec+.sec{border-top:1px solid var(--border);margin-top:var(--s4);padding-top:var(--s4)}
.eyebrow{font-size:var(--t-xs);text-transform:uppercase;letter-spacing:.08em;color:var(--muted);margin:0 0 var(--s2);font-weight:650}
.card h2{font-size:var(--t-xs);text-transform:uppercase;letter-spacing:.08em;color:var(--muted);margin:0 0 var(--s3);font-weight:650}
.seg{display:flex;background:var(--card2);border:1px solid var(--border);border-radius:10px;padding:3px;gap:3px}
.seg button{flex:1;border:0;border-radius:7px;padding:7px 0;background:none;color:var(--ink2);font-weight:600;cursor:pointer;transition:background var(--dur),color var(--dur)}
.seg button.on{background:var(--card);color:var(--ink);box-shadow:0 1px 3px rgba(0,0,0,.16)}
.grid{display:grid;grid-template-columns:1fr 1fr;gap:var(--s3)}.grid .wide{grid-column:1/-1}.sec .grid{margin-top:var(--s3)}
label.f{display:flex;flex-direction:column;gap:6px;font-size:var(--t-xs);color:var(--ink2);font-weight:600}
select,input.q{font-size:13.5px;padding:8px 10px;border:1px solid var(--border2);border-radius:var(--r-sm);background-color:var(--card);color:var(--ink);width:100%;font-weight:500;transition:border-color var(--dur)}
select:hover,input.q:hover{border-color:var(--muted)}
select{padding-right:30px;appearance:none;-webkit-appearance:none;cursor:pointer;background-image:linear-gradient(45deg,transparent 50%,var(--muted) 50%),linear-gradient(135deg,var(--muted) 50%,transparent 50%);background-position:calc(100% - 16px) 52%,calc(100% - 11px) 52%;background-size:5px 5px;background-repeat:no-repeat}
input.q{padding-left:34px;background-image:url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='16' height='16' viewBox='0 0 24 24' fill='none' stroke='%23807f78' stroke-width='2.4' stroke-linecap='round'%3E%3Ccircle cx='11' cy='11' r='7'/%3E%3Cpath d='M20 20l-3.6-3.6'/%3E%3C/svg%3E");background-repeat:no-repeat;background-position:11px 50%}
.tog{display:flex;align-items:center;gap:var(--s3);font-size:13px;color:var(--ink2);cursor:pointer;padding:4px 0}.tog+.tog{margin-top:var(--s2)}
.tog input{appearance:none;-webkit-appearance:none;margin:0;width:36px;height:21px;border-radius:999px;background:var(--track);border:1px solid var(--border2);position:relative;cursor:pointer;flex:none;transition:background var(--dur)}
.tog input::after{content:"";position:absolute;top:2px;left:2px;width:15px;height:15px;border-radius:50%;background:#fff;box-shadow:0 1px 2px rgba(0,0,0,.35);transition:left var(--dur) var(--ease)}
.tog input:checked{background:var(--acc);border-color:var(--acc)}.tog input:checked::after{left:17px}
.chips{display:flex;flex-wrap:wrap;gap:6px}
.chip{display:inline-flex;align-items:center;gap:6px;max-width:100%;padding:4px 6px 4px 10px;border-radius:999px;background:var(--accsoft);color:var(--acc-ink);font-size:var(--t-xs);font-weight:600;border:1px solid transparent}
.chip.fixed{padding-right:10px;background:var(--card2);color:var(--ink2);border-color:var(--border)}
.chip span{overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.chip button{width:18px;height:18px;border:0;border-radius:50%;background:transparent;color:inherit;cursor:pointer;display:grid;place-items:center;padding:0;flex:none}
.chip button:hover{background:rgba(42,120,214,.18)}
/* ---------- KPI strip ---------- */
#kpibar{padding:var(--s4) var(--s4) var(--s3);display:flex;flex-direction:column;gap:var(--s2);border-bottom:1px solid var(--border);background:var(--panel)}
.scope{font-size:var(--t-sm);color:var(--ink2);margin:0}.scope b{color:var(--ink)}
.kpis{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:var(--s3)}
.kpi{position:relative;background:var(--card);border:1px solid var(--border);border-radius:var(--r-lg);padding:var(--s3) var(--s4) var(--s3) var(--s4);box-shadow:var(--e1);overflow:hidden}
.kpi::before{content:"";position:absolute;left:0;top:14px;bottom:14px;width:3px;border-radius:0 3px 3px 0;background:var(--acc)}
.kpi .top{display:flex;align-items:center;justify-content:space-between;gap:var(--s2)}
.kpi .ic{width:28px;height:28px;border-radius:8px;background:var(--accsoft);color:var(--acc-ink);display:grid;place-items:center;flex:none}
.kpi .v{font-size:var(--t-3xl);font-weight:750;letter-spacing:-.025em;line-height:1.05;font-variant-numeric:tabular-nums}.kpi .v small{font-size:var(--t-md);font-weight:650;color:var(--ink2);margin-left:3px;letter-spacing:0}
.kpi .l{font-size:var(--t-xs);color:var(--ink2);margin-top:6px;line-height:1.3}
/* ---------- selected area & ranking ---------- */
.hint{color:var(--muted);font-size:13px;margin:0}
.empty{display:flex;gap:var(--s3);align-items:flex-start}.empty .ei{width:34px;height:34px;border-radius:10px;background:var(--card2);color:var(--muted);display:grid;place-items:center;flex:none}
.pop p{margin:8px 0}.pop .pt{font-size:var(--t-xl);display:block;line-height:1.2;letter-spacing:-.01em}.pop .where{color:var(--ink2);font-size:var(--t-sm)}.pop .pn{color:var(--muted);font-size:var(--t-xs)}
.pop .ps{background:var(--accsoft);border-radius:var(--r-md);padding:10px 12px;color:var(--ink)}.pop .hot{color:var(--warn);font-weight:650;font-size:var(--t-xs)}
.pop .lv{border-top:1px solid var(--border);padding-top:8px}.pop h4{font-size:var(--t-xs);margin:14px 0 6px;color:var(--ink2);font-weight:650;text-transform:uppercase;letter-spacing:.06em}
.dist{margin:10px 0 2px}.dist .strip{position:relative;display:flex;gap:2px;height:10px}.dist .strip i{flex:1;border-radius:2px}
.dist .mk{position:absolute;top:-5px;width:0;height:0;border-left:6px solid transparent;border-right:6px solid transparent;border-top:7px solid var(--ink);transform:translateX(-6px)}
.dist .cap{display:flex;justify-content:space-between;font-size:var(--t-xs);color:var(--muted);margin-top:4px}
.hb{display:flex;flex-direction:column;gap:2px}
.hr{display:grid;grid-template-columns:var(--lw,132px) 1fr 62px;align-items:center;gap:var(--s2);font-size:var(--t-sm);padding:4px 6px;border-radius:var(--r-sm);transition:background var(--dur)}
.hr[data-go]{cursor:pointer}.hr[data-go]:hover{background:var(--card2)}
.hl{overflow:hidden;text-overflow:ellipsis;white-space:nowrap;color:var(--ink2)}.hv{text-align:right;font-variant-numeric:tabular-nums;font-weight:650}
.rk{font-variant-numeric:tabular-nums;color:var(--muted);font-weight:650;margin-right:6px}
.ht{position:relative;height:10px;background:var(--track);border-radius:3px}
.hf{display:block;height:100%;border-radius:0 3px 3px 0;min-width:2px}.hs{display:flex;height:100%;gap:2px}.hs span{display:block;height:100%}.hs span:last-child{border-radius:0 3px 3px 0}
.href{position:absolute;top:-4px;bottom:-4px;width:2px;background:var(--ink);opacity:.65}
.grp{font-size:var(--t-sm);font-weight:650;color:var(--ink);margin:var(--s2) 6px 0}
.keys{display:flex;flex-wrap:wrap;gap:var(--s3);font-size:var(--t-sm);color:var(--ink2);margin:0 0 var(--s2)}.keys i{display:inline-block;width:12px;height:12px;border-radius:3px;margin-right:6px;vertical-align:-1px}
/* ---------- pages ---------- */
.page{overflow-y:auto;height:calc(100dvh - var(--hh,60px));width:100%}.wrap{max-width:1180px;margin:0 auto;padding:var(--s6) var(--s5) 56px}
.page h2.ttl{font-size:var(--t-2xl);margin:0 0 6px;letter-spacing:-.02em}.page p.lead{color:var(--ink2);margin:0 0 var(--s5);max-width:780px;font-size:15px}
.hero{display:grid;grid-template-columns:repeat(4,1fr);gap:var(--s3);margin-bottom:var(--s4)}
.fgrid{display:grid;grid-template-columns:repeat(2,1fr);gap:var(--s4)}
.fcard{padding:var(--s5)}.fcard h3{font-size:var(--t-lg);margin:0 0 4px;line-height:1.3;letter-spacing:-.01em}.fcard .sub{color:var(--ink2);font-size:var(--t-sm);margin:0 0 var(--s4)}.fcard .foot{color:var(--muted);font-size:var(--t-xs);margin:var(--s3) 0 0}
.span2{grid-column:1/-1}
.tabs{display:flex;gap:2px;border-bottom:1px solid var(--border);margin:-4px -4px var(--s3)}
.tabs button{font-size:13px;font-weight:650;border:0;background:none;color:var(--muted);padding:9px 12px;cursor:pointer;border-bottom:2px solid transparent;transition:color var(--dur)}
.tabs button.on{color:var(--ink);border-bottom-color:var(--acc)}
#panel{overflow-x:auto}#panel h3{font-size:13.5px;margin:var(--s3) 0 6px}
.about li{margin:8px 0;color:var(--ink2)}.about h3{font-size:var(--t-lg);margin:4px 0 var(--s2)}.about ol{padding-left:20px}#notes ul{padding-left:18px}
table.t{border-collapse:collapse;width:100%;font-size:12px;font-variant-numeric:tabular-nums}
table.t th{background:var(--card);color:var(--ink2);font-weight:650;border-bottom:1px solid var(--border2);padding:7px 6px;text-align:right;vertical-align:bottom}
table.t td{border-bottom:1px solid var(--border);padding:6px;text-align:right}table.t tbody tr:nth-child(even) td{background:var(--card2)}
table.t th:first-child,table.t td:first-child{text-align:left}
.n{color:var(--muted);font-size:var(--t-xs)}
/* ---------- legend & map chrome ---------- */
.legend{background:var(--card);color:var(--ink);border:1px solid var(--border);border-radius:var(--r-lg);box-shadow:var(--e2);font-size:12px;line-height:18px;width:290px;overflow:hidden}
.legend .lhead{display:flex;align-items:center;gap:var(--s2);width:100%;border:0;background:none;padding:10px 12px;cursor:pointer;text-align:left;color:var(--ink)}
.legend .lhead .lh{flex:1;font-weight:700;font-size:var(--t-sm);line-height:1.3}
.legend .chev{transition:transform var(--dur) var(--ease);color:var(--muted)}.legend.closed .chev{transform:rotate(-90deg)}
.legend .lbody{padding:0 12px 11px}.legend.closed .lbody{display:none}
.legend .lsub{color:var(--ink2);display:block}.legend .ld{display:block;color:var(--ink2);font-size:11px;line-height:14px;margin:3px 0 8px}.legend .lt{color:var(--muted);font-style:italic}
.legend .r{display:flex;align-items:center;gap:var(--s2);margin:1px -4px;padding:1px 4px;border-radius:6px}.legend .r i{display:inline-block;width:26px;height:12px;border-radius:3px;flex:none;box-shadow:inset 0 0 0 1px rgba(0,0,0,.06)}
.legend .x{border-top:1px solid var(--border);margin-top:8px;padding-top:8px;color:var(--ink2)}
.leaflet-tooltip.tt,#tip{background:var(--card);color:var(--ink);border:1px solid var(--border2);border-radius:10px;box-shadow:var(--e2);padding:7px 10px;font:var(--t-sm)/1.4 var(--font)}
.leaflet-tooltip.tt::before{display:none}.tt .tm,#tip .tm{color:var(--ink2)}
#tip{position:fixed;pointer-events:none;z-index:2000;display:none;max-width:280px}
.leaflet-container{font:inherit;background:var(--mapbg)}.leaflet-control-attribution{font-size:10.5px;background:color-mix(in srgb,var(--card) 85%,transparent)!important;color:var(--muted)}
.leaflet-control-attribution a{color:var(--acc-ink)}
.leaflet-bar{border:1px solid var(--border)!important;border-radius:10px!important;box-shadow:var(--e2)!important;overflow:hidden}
.leaflet-bar a{background:var(--card);color:var(--ink);border-bottom:1px solid var(--border)!important;width:34px!important;height:34px!important;line-height:34px!important}
.leaflet-bar a:hover{background:var(--card2)}
@media (max-width:1240px){.kpi .v{font-size:var(--t-2xl)}}
@media (max-width:980px){.hero{grid-template-columns:repeat(2,1fr)}.fgrid{grid-template-columns:1fr}.brand p{display:none}.kpis{grid-template-columns:repeat(2,minmax(0,1fr))}}
/* ---------- round 2: tools, states, motion ---------- */
.tools{display:flex;flex-direction:column}.tools button{width:34px;height:34px;display:grid;place-items:center;border:0;border-bottom:1px solid var(--border);background:var(--card);color:var(--ink);cursor:pointer;padding:0;border-radius:0}
.tools button:last-child{border-bottom:0}.tools button:hover{background:var(--card2)}.tools button[disabled]{color:var(--muted);opacity:.45;cursor:default}
.toast{position:fixed;left:50%;bottom:28px;transform:translate(-50%,16px);background:var(--ink);color:var(--page);padding:9px 16px;border-radius:999px;font-size:var(--t-sm);font-weight:650;box-shadow:var(--e2);opacity:0;pointer-events:none;transition:opacity var(--dur),transform var(--dur);z-index:3000;max-width:90vw;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.toast.on{opacity:1;transform:translate(-50%,0)}
.sk{position:relative;overflow:hidden;background:var(--card2);color:transparent;border-radius:6px;display:block}
.sk::after{content:"";position:absolute;inset:0;background:linear-gradient(90deg,transparent,rgba(255,255,255,.16),transparent);transform:translateX(-100%);animation:sh 1.2s infinite}
@keyframes sh{to{transform:translateX(100%)}}
#loading{position:absolute;inset:0;display:grid;place-items:center;z-index:800;background:var(--mapbg);color:var(--muted);font-size:var(--t-sm);text-align:center;transition:opacity 200ms}
#loading .spin{width:30px;height:30px;border-radius:50%;border:3px solid var(--track);border-top-color:var(--acc);animation:spin .8s linear infinite;margin:0 auto 10px}
@keyframes spin{to{transform:rotate(360deg)}}
.legend .r[data-cls]{cursor:default;outline:none}.legend .r[data-cls]:hover,.legend .r[data-cls]:focus-visible{background:var(--card2)}
#mapwrap:fullscreen{background:var(--mapbg)}
.leaflet-overlay-pane{transition:opacity 200ms ease}
.ptoggle,.handle{display:none}
.ptoggle{width:auto;padding:0 12px;gap:6px;font-weight:650;font-size:13px;height:34px;align-items:center}
.handle{align-items:center;justify-content:center;gap:10px;width:100%;border:1px solid var(--border);background:var(--card);border-radius:var(--r-md);padding:10px 14px;font-weight:650;cursor:pointer;flex:none;color:var(--ink)}
.handle .grip{width:36px;height:4px;border-radius:2px;background:var(--border2)}
@media (prefers-reduced-motion:reduce){html{scroll-behavior:auto}*,*::before,*::after{transition:none!important;animation:none!important}}
/* tablet: the panel slides over the map */
@media (max-width:1100px){#app{position:relative}
 #side{position:absolute;z-index:1100;top:0;left:0;bottom:0;width:min(390px,90vw);transform:translateX(-102%);transition:transform 220ms var(--ease);box-shadow:var(--e2)}
 #app.panel-open #side{transform:none}.ptoggle{display:inline-flex}.handle{display:flex}.handle .grip{display:none}}
/* phone: the panel is a bottom sheet */
@media (max-width:700px){
 header{height:auto;flex-wrap:wrap;padding:8px 12px;gap:var(--s2)}nav.pages{margin-left:0;width:100%;height:40px;order:3}nav.pages button{flex:1;padding:0 4px;font-size:13px}
 #kpibar{padding:var(--s2) var(--s3)}.kpis{grid-template-columns:repeat(2,minmax(0,1fr));gap:var(--s2)}.kpi{padding:var(--s2) var(--s3)}.kpi .v{font-size:var(--t-xl)}.kpi .ic{display:none}
 #side{position:fixed;top:auto;left:0;right:0;bottom:0;width:auto;height:86vh;border-right:0;border-radius:18px 18px 0 0;border-top:1px solid var(--border2);transform:translateY(calc(100% - 62px));transition:transform 220ms var(--ease)}
 #app.panel-open #side{transform:none}.ptoggle{display:none}.handle .grip{display:block}.handle{flex-direction:column;gap:6px;padding:8px 14px}
 #mapwrap{padding-bottom:62px}.legend{width:230px}.legend .ld{display:none}
 .page{height:auto}.wrap{padding:var(--s5) var(--s3) 90px}.hero{grid-template-columns:1fr 1fr}}
</style></head><body>
<header><div class="brand"><div class="logo" aria-hidden="true"><svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="#fff" stroke-width="2.2" stroke-linecap="round" stroke-linejoin="round"><path d="M3 10l9-6 9 6"/><path d="M5 10v9h14v-9"/><path d="M10 19v-5h4v5"/></svg></div>
<div style="min-width:0"><h1>School Access Explorer</h1><p>Government + Aided schools · India and Kerala · distance and seats for every 100 m cell</p></div></div>
<nav class="pages" id="pages" aria-label="Sections"><button data-p="app" class="on" aria-current="page">Explore map</button><button data-p="find">Key findings</button><button data-p="about">About the data</button><span class="ind" id="ind" aria-hidden="true"></span></nav>
<button class="iconbtn" id="theme" aria-label="Switch between light and dark theme" title="Light / dark theme"><svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" aria-hidden="true"><circle cx="12" cy="12" r="4.2"/><path d="M12 2v2.2M12 19.8V22M4.9 4.9l1.6 1.6M17.5 17.5l1.6 1.6M2 12h2.2M19.8 12H22M4.9 19.1l1.6-1.6M17.5 6.5l1.6-1.6"/></svg></button></header>
<div id="app" class="view on"><aside id="side" aria-label="Filters and details"><button class="handle" id="handle" aria-controls="side" aria-expanded="false"><span class="grip" aria-hidden="true"></span><span id="handlet">Filters &amp; details</span></button>
<section class="card" aria-label="Filters">
<div class="sec"><p class="eyebrow" id="h-where">Where</p><div class="seg" id="seg" role="group" aria-labelledby="h-where"><button data-v="in" class="on" aria-pressed="true">India</button><button data-v="k" aria-pressed="false">Kerala</button></div>
<select id="view" hidden aria-hidden="true"><option value="in">India</option><option value="k">Kerala</option></select>
<div class="grid"><label class="f wide" id="stw">State / UT<select id="state"></select></label><label class="f wide">Areas<select id="unit"></select></label></div></div>
<div class="sec"><p class="eyebrow">What</p><div class="grid" style="margin-top:0"><label class="f wide">School level<select id="level"></select></label><label class="f wide">Show on map<select id="measure"></select></label></div></div>
<div class="sec"><p class="eyebrow">Layers</p><label class="tog"><input type="checkbox" id="hot" checked>Outline poor-access clusters (Gi*)</label>
<label class="tog" id="prl"><input type="checkbox" id="pri" checked>Priority areas for new schools</label></div>
<div class="sec"><p class="eyebrow">Find</p><label class="f">Find an area<input class="q" id="q" list="ql" placeholder="Type a name, e.g. Wayanad" autocomplete="off"><datalist id="ql"></datalist></label><p class="hint" id="qmsg" role="status" style="margin-top:6px;display:none"></p></div>
<div class="sec"><p class="eyebrow">Active selection</p><div class="chips" id="chips" aria-live="polite"></div></div>
</section>
<section class="card" id="detail" aria-live="polite"></section>
<section class="card" id="rank"></section>
</aside><div id="main"><div id="kpibar"><div style="display:flex;align-items:center;gap:var(--s2)"><button class="iconbtn ptoggle" id="ptoggle" aria-controls="side" aria-expanded="false"><svg width="15" height="15" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.2" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M4 5h16l-6 8v5l-4 2v-7z"/></svg>Filters &amp; details</button><p class="scope" id="scope" style="flex:1"></p></div><div class="kpis" id="kpis" aria-busy="true"><div class="kpi"><span class="sk" style="height:30px;width:58%">&nbsp;</span><span class="sk" style="height:12px;width:86%;margin-top:10px">&nbsp;</span></div><div class="kpi"><span class="sk" style="height:30px;width:58%">&nbsp;</span><span class="sk" style="height:12px;width:86%;margin-top:10px">&nbsp;</span></div><div class="kpi"><span class="sk" style="height:30px;width:58%">&nbsp;</span><span class="sk" style="height:12px;width:86%;margin-top:10px">&nbsp;</span></div><div class="kpi"><span class="sk" style="height:30px;width:58%">&nbsp;</span><span class="sk" style="height:12px;width:86%;margin-top:10px">&nbsp;</span></div></div></div><div id="mapwrap"><div id="map" role="region" aria-label="Map of school access"></div><div id="loading" role="status"><div><div class="spin"></div>Loading map…</div></div></div></div></div>
<main id="find" class="view page"><div class="wrap">
<h2 class="ttl">Key findings</h2><p class="lead">What the 140 million population cells tell us about Government and Aided schools. All charts are computed from the same results as the map; click a state to open it on the map.</p>
<div class="hero" id="hero"></div><div class="fgrid" id="fgrid"></div>
<div class="card" style="margin-top:var(--s4)"><h2>Detailed tables</h2><div class="tabs" id="tabs" role="tablist"></div><div id="panel"></div></div>
</div></main>
<main id="about" class="view page"><div class="wrap about" id="aboutw"></div></main>
<div class="toast" id="toast" role="status" aria-live="polite"></div><div id="tip" role="tooltip"></div>
<script>__LEAFLET_JS__</script><script>
const D=__DATA__;
const LV=D.lv, UNITS={in:[["in_state","States"],["in_district","Districts"],["in_sub","Sub-districts"]],k:[["k_district","Districts"],["k_taluk","Taluks"],["k_lb","Local bodies"]]};
const LEVELS={in:["primary","upper_primary","secondary","higher_secondary"],k:["lp","up","hs","hss"]};
const MEAS={b:["% of children beyond the distance norm",1],n:["mean distance to nearest school (km)",1],m:["seats per child (norm-based)",-1],e:["seats per child (enrolment check)",-1],type:["area type (K-Means)",0]};
const LDESC={b:"Share of children who live farther than the distance norm from the nearest school of this level.",n:"Average straight-line distance from a child to the nearest school of this level.",m:"School seats a child can reach, shared with the other children nearby. 1 = a seat for every child.",e:"Same as seats per child, but seats = the pupils enrolled now (check run)."};
const PAL=["#cde2fb","#9ec5f4","#6da7ec","#3987e5","#256abf","#184f95","#0d366b"];
const CLUSTER="#eb6834";
const TYPEC={"Well served":"#1baf7a","Crowded: near but few places":"#eda100","Far from primary school":"#eb6834","Higher secondary far":"#2a78d6","Remote: far at every level":"#6250d6","Not classified":"#b4b3ac"};
const NORM={primary:"1 km",upper_primary:"3 km",secondary:"5 km",higher_secondary:"7 km",lp:"1 km",up:"3 km",hs:"5 km (3 km in hilly areas)",hss:"7 km"};
const SHORT={primary:"Primary",upper_primary:"Upper primary",secondary:"Secondary",higher_secondary:"Higher sec.",lp:"LP",up:"UP",hs:"HS",hss:"HSS"};
const TYPED={"Well served":"Schools are near and there are enough seats at most levels.","Crowded: near but few places":"Schools are close, but there are too few seats for the number of children.","Far from primary school":"Many young children live more than 1 km from a primary school.","Higher secondary far":"Lower levels are fine, but many teenagers live far from a higher secondary school.","Remote: far at every level":"Children live far from schools at every level.","Not classified":"Not grouped (too few children or no data)."};
const ICON={kids:'<path d="M9 7a3 3 0 1 0 6 0a3 3 0 1 0-6 0"/><path d="M5 21v-1a7 7 0 0 1 14 0v1"/>',far:'<path d="M12 21s-6-5.3-6-11a6 6 0 0 1 12 0c0 5.7-6 11-6 11z"/><circle cx="12" cy="10" r="2.2"/>',dist:'<path d="M4 18h16"/><path d="M4 14v8M20 14v8"/><path d="M8 18v-2M12 18v-3M16 18v-2"/>',seat:'<path d="M6 12V6a2 2 0 0 1 2-2h8a2 2 0 0 1 2 2v6"/><path d="M4 12h16v4H4z"/><path d="M6 16v4M18 16v4"/>',
 x:'<path d="M6 6l12 12M18 6L6 18"/>',click:'<path d="M9 9l11 4-5 1.5L13.5 20z"/><path d="M5 3v3M3 5h3M5.5 8.5L4 10"/>',list:'<path d="M8 6h12M8 12h12M8 18h12"/><path d="M4 6h.01M4 12h.01M4 18h.01"/>',home:'<path d="M3 11l9-7 9 7"/><path d="M5 10v10h14V10"/>',fit:'<path d="M4 9V4h5M20 9V4h-5M4 15v5h5M20 15v5h-5"/><rect x="9" y="9" width="6" height="6" rx="1"/>',full:'<path d="M8 3H3v5M16 3h5v5M8 21H3v-5M16 21h5v-5"/>',img:'<rect x="3" y="4" width="18" height="16" rx="2"/><circle cx="9" cy="10" r="2"/><path d="M21 17l-5-5-9 8"/>',link:'<path d="M10 14a4 4 0 0 0 5.7 0l3-3a4 4 0 0 0-5.7-5.7l-1 1"/><path d="M14 10a4 4 0 0 0-5.7 0l-3 3a4 4 0 0 0 5.7 5.7l1-1"/>'};
const RM=matchMedia("(prefers-reduced-motion: reduce)").matches;
const svg=(k,s)=>`<svg width="${s||16}" height="${s||16}" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true">${ICON[k]}</svg>`;
const $=id=>document.getElementById(id);
const cv=n=>getComputedStyle(document.documentElement).getPropertyValue(n).trim();
function fmt(x,d){return x==null||!isFinite(x)?"–":Number(x).toLocaleString(undefined,{maximumFractionDigits:d,minimumFractionDigits:d});}
Object.values(D.layers).forEach(fc=>fc.features.forEach(f=>{if(/^data not available$/i.test(f.properties.name||""))f.properties.name="Unnamed area";}));
const esc=s=>String(s).replace(/&/g,"&amp;").replace(/</g,"&lt;").replace(/"/g,"&quot;");

/* ---------- shared: children-weighted totals over a set of areas ---------- */
function agg(fs,lv,skipFlag){let C=0,B=0,CB=0,N=0,CN=0,M=0,CM=0;fs.forEach(f=>{const p=f.properties,c=p["c_"+lv];if(!c||(skipFlag&&p["f_"+lv]))return;C+=c;
 if(p["b_"+lv]!=null){B+=c*p["b_"+lv];CB+=c;}if(p["n_"+lv]!=null){N+=c*p["n_"+lv];CN+=c;}if(p["m_"+lv]!=null){M+=c*p["m_"+lv];CM+=c;}});
 return{c:C,b:CB?B/CB:null,n:CN?N/CN:null,m:CM?M/CM:null,far:CB?B/100:null};}
/* horizontal bars: rows [{label,value,col,tip,go,rank}] */
function hbars(rows,o){o=o||{};const mx=o.max||Math.max(...rows.map(r=>r.value||0))*1.08||1;
 return `<div class="hb" style="--lw:${o.lw||132}px">`+rows.map(r=>`<div class="hr"${r.go!=null?` data-go="${esc(r.go)}" tabindex="0" role="button" aria-label="${esc((r.aria||r.label)+": "+(o.f?o.f(r.value):fmt(r.value,1)))}"`:""} data-tip="${esc(r.tip||"")}"><span class="hl" title="${esc(r.aria||r.label)}">${r.rank?`<span class="rk">${r.rank}</span>`:""}${r.label}</span><span class="ht"><span class="hf" style="width:${Math.max(0,Math.min(100,100*(r.value||0)/mx))}%;background:${r.col||"var(--acc)"}"></span>${o.ref!=null?`<span class="href" style="left:${100*o.ref/mx}%"></span>`:""}</span><span class="hv">${o.f?o.f(r.value):fmt(r.value,1)}</span></div>`).join("")+`</div>`;}
document.addEventListener("mousemove",e=>{const t=e.target.closest&&e.target.closest("[data-tip]"),el=$("tip");if(!t||!t.dataset.tip){el.style.display="none";return;}
 el.innerHTML=t.dataset.tip;el.style.display="block";const w=el.offsetWidth;el.style.left=Math.min(innerWidth-w-8,e.clientX+14)+"px";el.style.top=(e.clientY+14)+"px";});

/* ---------- map page ---------- */
const map=L.map("map",{preferCanvas:true,zoomSnap:0.25,zoomControl:true});
map.attributionControl.addAttribution("Boundaries: geoBoundaries (CC BY 4.0) · Kerala local bodies © OpenStreetMap contributors (ODbL) · Population: WorldPop (CC BY 4.0)");
let geo=null,pri=null,border=null,gaps=null,states=null,sel=null,LAY=[],CUR={},legend=L.control({position:"bottomright"});legend.onAdd=()=>{const d=L.DomUtil.create("div","legend");L.DomEvent.disableClickPropagation(d);L.DomEvent.disableScrollPropagation(d);return d;};legend.addTo(map);
let legendClosed=innerWidth<=700,hl=null;  // compact legend on phones
const Tools=L.Control.extend({options:{position:"topleft"},onAdd(){const d=L.DomUtil.create("div","leaflet-bar tools");
 d.innerHTML=[["reset","Reset view","home"],["fit","Fit to selection","fit"],["full","Fullscreen","full"],["png","Export current view as PNG","img"],["link","Copy link to this view","link"]].map(([id,t,ic])=>`<button type="button" id="t-${id}" aria-label="${t}" title="${t}">${svg(ic,16)}</button>`).join("");
 L.DomEvent.disableClickPropagation(d);return d;}});
map.addControl(new Tools());
const LEG=document.querySelector(".legend");
LEG.addEventListener("mouseover",e=>{const r=e.target.closest(".r[data-cls]");if(r){if(r!==hl){hl=r;highlight(r.dataset.cls);}}else if(hl){hl=null;unhighlight();}});
LEG.addEventListener("mouseleave",()=>{if(hl){hl=null;unhighlight();}});
LEG.addEventListener("focusin",e=>{const r=e.target.closest(".r[data-cls]");if(r){hl=r;highlight(r.dataset.cls);}});
LEG.addEventListener("focusout",()=>{if(hl){hl=null;unhighlight();}});
function highlight(k){LAY.forEach(it=>{const on=it.cls==k;it.l.setStyle({fillOpacity:on?.94:.12,opacity:on?.9:.15});});}
function unhighlight(){LAY.forEach(it=>it.g.resetStyle(it.l));if(sel)sel.l.setStyle({weight:3,color:"#0b0b0b"});}
function fill(s,opts,keep){const v=s.value;s.innerHTML=opts.map(([k,t])=>`<option value="${k}">${t}</option>`).join("");if(keep&&opts.some(o=>o[0]==v))s.value=v;}
const STATES=D.layers.in_state.features.map(f=>f.properties.name).sort((a,b)=>a.localeCompare(b));
fill($("state"),[["","All India"],...STATES.map(s=>[s,s])],false);
const curState=()=>$("view").value=="in"?$("state").value:"";
const units=()=>curState()?UNITS.in.slice(1):UNITS[$("view").value];  // one state: its districts or sub-districts
function setState(){fill($("unit"),units(),true);setUnit(true);}
function setView(){const v=$("view").value;document.querySelectorAll("#seg button").forEach(b=>{b.classList.toggle("on",b.dataset.v==v);b.setAttribute("aria-pressed",b.dataset.v==v);});$("stw").style.display=v=="in"?"":"none";$("state").value="";fill($("unit"),units(),false);$("unit").value=UNITS[v][1][0];fill($("level"),LEVELS[v].map(l=>[l,LV[l]]),false);$("prl").style.display=v=="k"?"":"none";resetDetail();setUnit(true);}
function msOf(u){let ms=["b","n","m","e"];if(u=="in_sub")ms=["type","b","n","m","e"];if(u=="k_taluk")ms.push("type");return ms;}
function setUnit(fit){const u=$("unit").value;const ms=msOf(u);
 fill($("measure"),ms.map(m=>[m,MEAS[m][0]]),true);if(u=="in_sub")$("measure").value="type";draw(fit);}
function breaks(vals){vals=vals.filter(x=>x!=null&&isFinite(x)).sort((a,b)=>a-b);if(!vals.length)return[];const b=[];for(let i=1;i<7;i++)b.push(vals[Math.floor(i*vals.length/7)]);return[...new Set(b)];}
function cls(x,b){let i=0;while(i<b.length&&x>=b[i])i++;return i;}
function val(p,m,lv){if(m=="type")return p.type||"Not classified";const x=p[m+"_"+lv];if(p["f_"+lv]||x==null)return"no data";return m=="b"?fmt(x,1)+" %":m=="n"?fmt(x,2)+" km":fmt(x,2);}
function tip(p,u,m,lv){return `<b>${p.name}</b>`+(p.district&&u!="k_district"?`<br><span class="tm">${p.district}</span>`:"")+`<br><span class="tm">${m=="type"?"Area type":MEAS[m][0]}:</span> <b>${val(p,m,lv)}</b>`;}
function label(p,u){return p.name+(u=="in_state"?"":" — "+(u.startsWith("k")?(p.district||"Kerala"):[p.district,p.state].filter(Boolean).join(", ")));}
function resetDetail(){$("detail").innerHTML=`<h2>Selected area</h2><div class="empty"><div class="ei">${svg("click",18)}</div><p class="hint">Click an area on the map, or use “Find an area”, to read what we found there.</p></div>`;}
function distStrip(p){const c=CUR;if(!c.m||c.m=="type"||!c.b)return"";const x=p[c.m+"_"+c.lv];if(x==null||p["f_"+c.lv])return"";
 const ci=c.colourIndex(x),n=PAL.length,pos=(ci+.5)/n*100,lab=val(p,c.m,c.lv);
 return `<div class="dist" aria-label="Position among all areas: ${esc(lab)}"><h4 style="margin-top:4px">Where it sits on the map scale · ${MEAS[c.m][0]}: ${lab}</h4><div class="strip">${PAL.map(col=>`<i style="background:${col}"></i>`).join("")}<span class="mk" style="left:${pos}%"></span></div><div class="cap"><span>good</span><span>poor</span></div></div>`;}
function choose(i,zoom){const it=LAY[i];if(!it)return;if(sel&&sel.l!==it.l)sel.g.resetStyle(sel.l);sel={l:it.l,g:it.g,i:i};updateTools();it.l.setStyle({weight:3,color:"#0b0b0b"});it.l.bringToFront();
 if(zoom)map.fitBounds(it.l.getBounds(),{padding:[60,60],maxZoom:$("view").value=="k"?11:9});
 const p=it.f.properties,body=popup(p,$("unit").value,$("level").value);
 $("detail").innerHTML="<h2>Selected area</h2>"+body.replace('</span>',"</span>"+distStrip(p));writeHash();
 if(innerWidth<=1100){setPanel(true);setTimeout(()=>$("detail").scrollIntoView({behavior:RM?"auto":"smooth",block:"start"}),240);}}
function bind(grp,u,m,lv){return(f,l)=>{const i=LAY.length;LAY.push({f:f,l:l,g:null,grp:grp});
 l.bindTooltip(()=>tip(f.properties,u,m,lv),{sticky:true,className:"tt",direction:"top",offset:[0,-8]});
 l.on("mouseover",()=>{if(!sel||sel.l!==l)l.setStyle({weight:2,color:"#0b0b0b"});});l.on("mouseout",()=>{if(!sel||sel.l!==l)grp().resetStyle(l);});
 l.on("click",()=>choose(i,false));};}
let KPREV={};
function countUp(root){root.querySelectorAll(".num[data-to]").forEach(el=>{const k=el.dataset.k,to=+el.dataset.to,d=+el.dataset.d,from=KPREV[k];KPREV[k]=to;if(RM||from==null||from===to)return;  /* first view shows the exact numbers at once (also for screenshots); later changes count up */const t0=performance.now(),T=450;
 (function step(t){const q=Math.min(1,Math.max(0,(t-t0)/T)),e=1-Math.pow(1-q,3);el.textContent=fmt(from+(to-from)*e,d);if(q<1)requestAnimationFrame(step);else el.textContent=fmt(to,d);})(t0);setTimeout(()=>{el.textContent=fmt(to,d);},T+80);});}  // the exact value always lands, even if frames pause
function kpiCard(ic,x,s,l,key,num,d){return `<div class="kpi"><div class="top"><div class="v"><span class="num"${key&&num!=null&&isFinite(num)?` data-k="${key}" data-to="${num}" data-d="${d}"`:""}>${x}</span><small>${s}</small></div><div class="ic">${svg(ic,16)}</div></div><div class="l" title="${esc(l)}">${l}</div></div>`;}
function kpis(lv){const v=$("view").value,st=curState();
 const fs=v=="k"?D.layers.k_district.features:(st?D.layers.in_state.features.filter(f=>f.properties.name==st):D.layers.in_state.features);
 const a=agg(fs,lv,false),C=a.c,big=C>=1e6?[C/1e6,1,"M"]:C>=1e3?[C/1e3,0,"k"]:[C,0,""];
 $("scope").innerHTML=`Showing <b>${v=="k"?"Kerala":(st||"All India")}</b> · ${LV[lv]}`;
 $("kpis").innerHTML=kpiCard("kids",fmt(big[0],big[1]),big[2],"children who need a Govt + Aided school","c",big[0],big[1])+kpiCard("far",fmt(a.b,1),"%",`live more than ${NORM[lv]} from a school`,"b",a.b,1)
  +kpiCard("dist",fmt(a.n,2),"km","average distance to the nearest school","n",a.n,2)+kpiCard("seat",fmt(a.m,2),"","seats per child (1 = a seat for every child)","m",a.m,2);$("kpis").removeAttribute("aria-busy");countUp($("kpis"));}
function chips(){const v=$("view").value,st=curState(),u=$("unit"),lv=$("level"),m=$("measure");const txt=s=>s.options[s.selectedIndex]?s.options[s.selectedIndex].text:"";
 const c=(t,act,lab)=>act?`<span class="chip"><span>${t}</span><button data-x="${act}" aria-label="Remove: ${esc(lab||t)}" title="Remove">${svg("x",12)}</button></span>`:`<span class="chip fixed"><span>${t}</span></span>`;
 let h=c(v=="k"?"Kerala":"India");if(st)h+=c(st,"state");h+=c(txt(u));h+=c(txt(lv),lv.selectedIndex>0?"level":"");h+=c(txt(m),m.selectedIndex>0?"measure":"");
 if($("hot").checked)h+=c("Gi* clusters","hot","Outline poor-access clusters (Gi*)");if(v=="k"&&$("pri").checked)h+=c("Priority areas","pri","Priority areas for new schools");$("chips").innerHTML=h;}
function ranking(m,lv,u){const el=$("rank");if(m=="type"){const cnt={};LAY.forEach(it=>{const t=it.f.properties.type||"Not classified";cnt[t]=(cnt[t]||0)+1;});
  el.innerHTML="<h2>Area types here</h2>"+hbars(Object.keys(TYPEC).filter(k=>cnt[k]).map(k=>({label:k,value:cnt[k],col:TYPEC[k],tip:`<b>${k}</b><br>${TYPED[k]}`})),{f:x=>fmt(x,0),lw:190});return;}
 const key=m+"_"+lv,dir=MEAS[m][1],rows=LAY.map((it,i)=>({i:i,p:it.f.properties})).filter(r=>r.p[key]!=null&&!r.p["f_"+lv]&&r.p["c_"+lv]>=100);
 rows.sort((a,b)=>dir>0?b.p[key]-a.p[key]:a.p[key]-b.p[key]);const top=rows.slice(0,10);
 const f=m=="b"?x=>fmt(x,1)+" %":m=="n"?x=>fmt(x,2)+" km":x=>fmt(x,2);
 el.innerHTML=`<h2>${dir>0?"Highest":"Lowest"} 10 here: ${MEAS[m][0]}</h2>`+(top.length?hbars(top.map((r,k)=>({label:r.p.name,value:r.p[key],go:r.i,rank:k+1,col:dir>0?"#184f95":"#256abf",tip:tip(r.p,u,m,lv)})),{f:f,lw:160,ref:m=="m"||m=="e"?1:null})
  :`<div class="empty"><div class="ei">${svg("list",18)}</div><p class="hint">No areas with data.</p></div>`)
  +`<p class="hint" style="margin-top:8px;font-size:var(--t-xs)">${dir>0?"Worst first":"Fewest seats first"} · areas with at least 100 children · click a name to zoom${m=="m"||m=="e"?" · line = 1 seat per child":""}</p>`;}
function draw(fit){const u=$("unit").value,lv=$("level").value,m=$("measure").value,all=D.layers[u],st=curState();const key=m+"_"+lv,dir=MEAS[m][0]?MEAS[m][1]:0;
 const fc=st?{type:"FeatureCollection",features:all.features.filter(f=>f.properties.state==st)}:all;
 const ok=f=>!(f.properties["f_"+lv]);const vals=m=="type"?[]:all.features.filter(ok).map(f=>f.properties[key]);const b=breaks(vals);  // all-India classes, also for one state
 const colourIndex=x=>{let c=cls(x,b);c=Math.round(c*(PAL.length-1)/Math.max(b.length,1));return dir<0?PAL.length-1-c:c;};
 const NODATA=cv("--nodata"),edge=cv("--edge"),stl=cv("--stateline");TYPEC["Not classified"]=NODATA;
 const colour=f=>{const p=f.properties;if(m=="type")return TYPEC[p.type]||NODATA;if(!ok(f)||p[key]==null)return NODATA;return PAL[colourIndex(p[key])];};
 CUR={u:u,lv:lv,m:m,b:b,dir:dir,colourIndex:colourIndex};
 const hot=f=>$("hot").checked&&(f.properties["hf_"+lv]||f.properties["hp_"+lv]);
 const ring=m=="type"?"#0b0b0b":CLUSTER,fine=u=="in_sub"||u=="k_lb";sel=null;LAY=[];
 if(geo)map.removeLayer(geo);
 geo=L.geoJSON(fc,{style:f=>({fillColor:colour(f),fillOpacity:.94,color:hot(f)?ring:edge,opacity:hot(f)?1:.9,weight:hot(f)?(fine?1.1:2.4):(fine?.25:.55)}),onEachFeature:bind(()=>geo,u,m,lv)}).addTo(map);
 if(gaps){map.removeLayer(gaps);gaps=null;}
 if(u=="k_lb")gaps=L.geoJSON(D.gaps,{style:{fillColor:"#9e9e9e",fillOpacity:.55,color:"#52514e",weight:1.2,dashArray:"4 3"},onEachFeature:bind(()=>gaps,u,m,lv)}).addTo(map);
 LAY.forEach(it=>{it.g=it.grp();const p=it.f.properties;it.cls=m=="type"?(p.type||"Not classified"):((!p["f_"+lv]&&p[key]!=null)?String(colourIndex(p[key])):"nd");});
 if(states){map.removeLayer(states);states=null;}
 if(u!="in_state"&&$("view").value=="in"&&!st)states=L.geoJSON(D.layers.in_state,{style:{fill:false,color:stl,weight:.8},interactive:false}).addTo(map);
 if(border){map.removeLayer(border);border=null;}
 if(st)border=L.geoJSON(D.layers.in_state.features.find(f=>f.properties.name==st),{style:{fill:false,color:cv("--ink"),weight:2.2},interactive:false}).addTo(map);
 if(fit)map.fitBounds((border||geo).getBounds(),{padding:[20,20],animate:!INIT});
 if(pri){map.removeLayer(pri);pri=null;}
 if($("view").value=="k"&&$("pri").checked){pri=L.layerGroup(D.pts.filter(p=>p.level==lv).map(p=>L.circleMarker([p.lat,p.lon],{radius:3+Math.sqrt(p.children)/6,color:"#0b0b0b",weight:1,fillColor:"#e87ba4",fillOpacity:.7})
   .bindTooltip(`<b>Priority area · ${LV[lv]}</b><br>${fmt(p.children,0)} children with no ${LV[lv]} school within the norm<br><span class="tm">${p.cells} cells of 100 m · ${p.local_body}, ${p.district}</span>`,{className:"tt",direction:"top"}))).addTo(map);}
 $("ql").innerHTML=LAY.map(it=>`<option value="${esc(label(it.f.properties,u))}">`).join("");$("q").value="";$("qmsg").style.display="none";
 legendHtml(m,b,dir,lv);kpis(lv);ranking(m,lv,u);chips();hl=null;
 const pane=map.getPane("overlayPane");if(!RM){pane.style.transition="none";pane.style.opacity=".3";requestAnimationFrame(()=>requestAnimationFrame(()=>{pane.style.transition="";pane.style.opacity="1";}));}
 writeHash();updateTools();}
function legendHtml(m,b,dir,lv){const el=document.querySelector(".legend");const NODATA=cv("--nodata");const r=(c,t,k)=>`<div class="r"${k!=null?` data-cls="${k}" tabindex="0"`:""}><i style="background:${c}"></i><span>${t}</span></div>`;
 const extra=($("hot").checked&&["in_sub","k_lb"].includes($("unit").value)?r("transparent;outline:2.5px solid "+(m=="type"?"#0b0b0b":CLUSTER)+";outline-offset:-2.5px","outline: part of a poor-access cluster (Gi*)"):"")
  +($("unit").value=="k_lb"?r("#9e9e9e;outline:1px dashed #52514e","no local-body boundary in OSM"):"")
  +($("view").value=="k"&&$("pri").checked?r("#e87ba4;border-radius:50%;width:13px;margin:0 6px 0 7px","priority area for new schools (size = children)"):"");
 const head=t=>`<button class="lhead" id="lhead" aria-expanded="${!legendClosed}" aria-controls="lbody"><span class="lh">${t}</span><svg class="chev" width="14" height="14" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.4" stroke-linecap="round" aria-hidden="true"><path d="M6 9l6 6 6-6"/></svg></button>`;
 let body;
 if(m=="type"){body=`<span class="ld">Sub-districts grouped by the kind of school-access problem they have.</span>`
  +Object.entries(TYPEC).map(([k,c])=>`<div class="r" data-cls="${esc(k)}" tabindex="0"><i style="background:${c}"></i><span><b>${k}</b></span></div><span class="ld" style="margin:0 0 4px 34px">${TYPED[k]||""}</span>`).join("")+(extra?`<div class="x">${extra}</div>`:"");
  el.innerHTML=head("Area type (K-Means)")+`<div class="lbody" id="lbody">${body}</div>`;}
 else{const lo=[null,...b],hi=[...b,null];const rows=lo.map((x,i)=>{let c=Math.round(i*(PAL.length-1)/Math.max(b.length,1));c=dir<0?PAL.length-1-c:c;const d=m=="b"?1:(m=="n"?2:3);
  const tag=c==0?" <span class=lt>good</span>":(c==PAL.length-1?" <span class=lt>poor</span>":"");
  return r(PAL[c],(x==null?"&lt; "+fmt(hi[i],d):(hi[i]==null?"≥ "+fmt(x,d):fmt(x,d)+" – "+fmt(hi[i],d)))+tag,c);});
  body=`<span class="lsub">${LV[lv]}</span><span class="ld">${LDESC[m]}</span>`+rows.join("")
  +`<div class="x">${r(NODATA,"no data (school list incomplete or too few children)","nd")}${extra}<div style="margin-top:3px;color:var(--muted)">light = good, dark = poor · 7 equal-count groups${curState()?" · all-India scale":""}</div></div>`;
  el.innerHTML=head(MEAS[m][0])+`<div class="lbody" id="lbody">${body}</div>`;}
 el.classList.toggle("closed",legendClosed);
 $("lhead").onclick=()=>{legendClosed=!legendClosed;el.classList.toggle("closed",legendClosed);$("lhead").setAttribute("aria-expanded",!legendClosed);};}
function seatWord(x){return x>=1?"enough seats":x>=0.8?"slightly too few seats":x>=0.5?"too few seats":"far too few seats";}
function farWord(b){return b<5?"almost every child is within reach":b<20?"most children are within reach":b<50?"many children live too far":"most children live too far";}
function popup(p,u,lv){const ls=LEVELS[u.startsWith("k")?"k":"in"];
 let h=`<div class="pop"><b class="pt">${p.name}</b><span class="where">`+[p.district,(p.state&&u!="in_state")?p.state:""].filter(Boolean).join(", ")+`</span>`;
 if(p.note)h+=`<p class="pn">${p.note}</p>`;
 if(p.type)h+=`<p><b>Area type: ${p.type}.</b> ${TYPED[p.type]||""}</p>`;
 const ok=ls.filter(l=>!p["f_"+l]&&p["b_"+l]!=null);
 if(ok.length){const far=ok.reduce((a,l)=>p["b_"+l]>p["b_"+a]?l:a),ms=ok.filter(l=>p["m_"+l]!=null),few=ms.length?ms.reduce((a,l)=>p["m_"+l]<p["m_"+a]?l:a):null;
  let s=p["b_"+far]>=20?`the biggest distance problem is at <b>${LV[far]}</b>: ${fmt(p["b_"+far],0)} % of children live more than ${NORM[far]} from the nearest school.`:"at every level, most children live within the distance norm.";
  if(few){const m=p["m_"+few];s+=m<0.95?` The fewest seats are at <b>${LV[few]}</b>: ${m<0.05?"less than 1 seat":"about "+fmt(m*10,0)+(fmt(m*10,0)=="1"?" seat":" seats")} for every 10 children.`:(m<1?` Seats are slightly fewer than children at <b>${LV[few]}</b>.`:" There are enough seats at every level.");}
  h+=`<p class="ps"><b>In short:</b> ${s}</p>`;
  h+=`<h4>Children beyond the distance norm</h4>`+hbars(ok.map(l=>({label:SHORT[l]+" ("+NORM[l].split(" ")[0]+" km)",value:p["b_"+l],col:"#184f95",tip:`${LV[l]}: ${fmt(p["b_"+l],1)} % beyond ${NORM[l]}`})),{max:100,f:x=>fmt(x,1)+" %",lw:120});
  if(ms.length)h+=`<h4>Seats per child (line = 1 seat for every child)</h4>`+hbars(ms.map(l=>({label:SHORT[l],value:p["m_"+l],col:"#3987e5",tip:`${LV[l]}: ${fmt(p["m_"+l],2)} seats per child`})),{max:Math.max(2,...ms.map(l=>p["m_"+l]))*1.05,ref:1,f:x=>fmt(x,2),lw:120});}
 h+=ls.map(l=>{const c=p["c_"+l],b=p["b_"+l],n=p["n_"+l],m=p["m_"+l];let t=`<p class="lv"><b>${LV[l]}</b>`;
  if(p["hf_"+l])t+=` · <span class="hot">in a cluster of areas where children live far</span>`;
  if(p["hp_"+l])t+=` · <span class="hot">in a cluster of areas with few seats</span>`;
  if(p["f_"+l]||b==null)return t+"<br>No data for this level.</p>";
  t+=`<br>About ${fmt(c,0)} children. ${fmt(b,1)} % live more than ${NORM[l]} from the nearest school (on average ${fmt(n,1)} km away): ${farWord(b)}.`;
  t+=m==null?" No school of this level is within reach, so no seats.":` ${fmt(m,2)} seats per child: ${seatWord(m)}.`;return t+"</p>";}).join("");
 if(p.types)h+="<p>Its sub-districts by area type: "+Object.entries(p.types).map(([k,v])=>`${k} ${v}`).join(", ")+".</p>";
 h+=`<p class="pn">Children = those expected in Government + Aided schools. Seats per child: 1 = a seat for every child. Distances are straight-line, so real journeys are longer.</p></div>`;return h;}
$("rank").addEventListener("click",e=>{const r=e.target.closest("[data-go]");if(r)choose(+r.dataset.go,true);});
$("rank").addEventListener("keydown",e=>{const r=e.target.closest("[data-go]");if(r&&(e.key=="Enter"||e.key==" ")){e.preventDefault();choose(+r.dataset.go,true);}});
$("q").addEventListener("change",()=>{const u=$("unit").value,raw=$("q").value.trim(),v=raw.toLowerCase();if(!v)return;
 let i=LAY.findIndex(it=>label(it.f.properties,u).toLowerCase()==v);if(i<0)i=LAY.findIndex(it=>it.f.properties.name.toLowerCase().startsWith(v));
 const msg=$("qmsg");if(i>=0){msg.style.display="none";choose(i,true);}else{msg.textContent=`No area called “${raw}” in this view. Try another name, or change “Areas”.`;msg.style.display="block";}});
$("chips").addEventListener("click",e=>{const b=e.target.closest("[data-x]");if(!b)return;const x=b.dataset.x;
 if(x=="state"){$("state").value="";setState();}else if(x=="level"){$("level").selectedIndex=0;draw(false);}else if(x=="measure"){$("measure").selectedIndex=0;draw(false);}
 else if(x=="hot"){$("hot").checked=false;draw(false);}else if(x=="pri"){$("pri").checked=false;draw(false);}});
$("seg").onclick=e=>{const v=e.target.dataset.v;if(!v||v==$("view").value)return;$("view").value=v;setView();};
$("state").onchange=setState;$("unit").onchange=()=>{resetDetail();setUnit(false);};$("level").onchange=()=>draw(false);$("measure").onchange=()=>draw(false);$("hot").onchange=()=>draw(false);$("pri").onchange=()=>draw(false);

/* ---------- map tools: reset, fit, fullscreen, PNG, link ---------- */
function fitScope(){map.fitBounds((border||geo).getBounds(),{padding:[20,20]});}
function updateTools(){const b=$("t-fit");if(b)b.disabled=!sel;}
$("t-reset").onclick=fitScope;
$("t-fit").onclick=()=>{if(sel)map.fitBounds(sel.l.getBounds(),{padding:[60,60],maxZoom:$("view").value=="k"?11:9});};
$("t-full").onclick=()=>{const w=$("mapwrap");if(document.fullscreenElement)document.exitFullscreen();else if(w.requestFullscreen)w.requestFullscreen();};
document.addEventListener("fullscreenchange",()=>{setTimeout(()=>map.invalidateSize(),60);$("t-full").setAttribute("aria-pressed",!!document.fullscreenElement);});
$("t-png").onclick=exportPNG;$("t-link").onclick=copyLink;
let tt;function toast(m){const e=$("toast");e.textContent=m;e.classList.add("on");clearTimeout(tt);tt=setTimeout(()=>e.classList.remove("on"),1900);}
function exportPNG(){const r=map.getRenderer(geo),c=r&&r._container;if(!c){toast("Nothing to export yet");return;}
 const dpr=window.devicePixelRatio||1,mr=$("map").getBoundingClientRect(),cr=c.getBoundingClientRect(),top=70,bot=28;
 const W=Math.round(mr.width*dpr),H=Math.round(mr.height*dpr),o=document.createElement("canvas");o.width=W;o.height=H+Math.round((top+bot)*dpr);const g=o.getContext("2d");
 g.fillStyle=cv("--card");g.fillRect(0,0,o.width,o.height);g.fillStyle=cv("--mapbg");g.fillRect(0,top*dpr,W,H);
 g.save();g.beginPath();g.rect(0,top*dpr,W,H);g.clip();g.drawImage(c,(cr.left-mr.left)*dpr,(cr.top-mr.top+top)*dpr,cr.width*dpr,cr.height*dpr);g.restore();
 const F=cv("--font")||"sans-serif";g.setTransform(dpr,0,0,dpr,0,0);g.fillStyle=cv("--ink");g.font="700 18px "+F;g.fillText("School Access Explorer",16,28);
 g.font="500 13px "+F;g.fillStyle=cv("--ink2");g.fillText($("scope").textContent+" · "+(CUR.m=="type"?"Area type (K-Means)":MEAS[CUR.m][0]),16,50);
 const rows=[...document.querySelectorAll(".legend .lbody .r")].map(r=>({c:getComputedStyle(r.querySelector("i")).backgroundColor,t:r.querySelector("span").textContent.trim()}));
 if(rows.length){const lw=300,lh=18,lx=mr.width-lw-14,ly=top+mr.height-rows.length*lh-46;g.fillStyle=cv("--card");g.strokeStyle=cv("--border2");g.beginPath();if(g.roundRect)g.roundRect(lx,ly,lw,rows.length*lh+34,10);else g.rect(lx,ly,lw,rows.length*lh+34);g.fill();g.stroke();
  g.fillStyle=cv("--ink");g.font="700 12px "+F;g.fillText(CUR.m=="type"?"Area type (K-Means)":MEAS[CUR.m][0],lx+10,ly+19);g.font="500 11.5px "+F;
  rows.forEach((r,i)=>{const y=ly+27+i*lh;g.fillStyle=r.c;g.fillRect(lx+10,y+2,22,11);g.fillStyle=cv("--ink2");g.fillText(r.t.length>46?r.t.slice(0,45)+"…":r.t,lx+40,y+12);});}
 g.font="500 10.5px "+F;g.fillStyle=cv("--muted");g.fillText("Boundaries: geoBoundaries (CC BY 4.0) · Kerala local bodies © OpenStreetMap contributors (ODbL) · Population: WorldPop (CC BY 4.0)",16,top+mr.height+18);
 o.toBlob(b=>{if(!b){toast("Export failed");return;}const a=document.createElement("a"),nm=($("scope").textContent+" "+(CUR.m=="type"?"area type":MEAS[CUR.m][0])).toLowerCase().replace(/[^a-z0-9]+/g,"-").replace(/^-|-$/g,"");
  a.href=URL.createObjectURL(b);a.download="school-access-"+nm+".png";document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(a.href),3000);toast("PNG saved");},"image/png");}
function writeHash(){const H=new URLSearchParams();H.set("v",$("view").value);const st=curState();if(st)H.set("st",st);H.set("u",$("unit").value);H.set("lv",$("level").value);H.set("m",$("measure").value);H.set("hot",$("hot").checked?1:0);
 if($("view").value=="k")H.set("pri",$("pri").checked?1:0);if(sel&&LAY[sel.i])H.set("sel",label(LAY[sel.i].f.properties,$("unit").value));try{history.replaceState(null,"","#"+H.toString());}catch(e){}}
function applyHash(h){let H;try{H=new URLSearchParams((h||"").slice(1));}catch(e){return;}if(!H.get("u"))return;const has=(s,v)=>[...s.options].some(o=>o.value==v);
 if(H.get("v")=="k"&&$("view").value!="k"){$("view").value="k";setView();}
 if($("view").value=="in"&&H.get("st")&&STATES.includes(H.get("st"))){$("state").value=H.get("st");fill($("unit"),units(),true);}
 if(has($("unit"),H.get("u")))$("unit").value=H.get("u");const u=$("unit").value;fill($("measure"),msOf(u).map(m=>[m,MEAS[m][0]]),false);if(u=="in_sub")$("measure").value="type";
 if(has($("level"),H.get("lv")))$("level").value=H.get("lv");if(has($("measure"),H.get("m")))$("measure").value=H.get("m");
 if(H.has("hot"))$("hot").checked=H.get("hot")=="1";if(H.has("pri"))$("pri").checked=H.get("pri")=="1";
 const s=H.get("sel");draw(!s);if(s){const i=LAY.findIndex(it=>label(it.f.properties,u)==s);if(i>=0)choose(i,true);else fitScope();}}
function copyLink(){writeHash();const u=location.href;if(navigator.clipboard&&window.isSecureContext)navigator.clipboard.writeText(u).then(()=>toast("Link copied"),()=>fallbackCopy(u));else fallbackCopy(u);}
function fallbackCopy(u){const t=document.createElement("textarea");t.value=u;t.setAttribute("readonly","");t.style.position="fixed";t.style.opacity="0";document.body.appendChild(t);t.select();let ok=false;try{ok=document.execCommand("copy");}catch(e){}t.remove();toast(ok?"Link copied":"Copy this link: "+u);}
function setPanel(open){$("app").classList.toggle("panel-open",open);["ptoggle","handle"].forEach(id=>$(id).setAttribute("aria-expanded",open));$("handlet").textContent=open&&innerWidth>700&&innerWidth<=1100?"Close":"Filters & details";setTimeout(()=>map.invalidateSize(),240);}
$("ptoggle").onclick=()=>setPanel(!$("app").classList.contains("panel-open"));$("handle").onclick=()=>setPanel(!$("app").classList.contains("panel-open"));
document.addEventListener("keydown",e=>{if(e.key=="Escape"&&$("app").classList.contains("panel-open")&&innerWidth<=1100)setPanel(false);});
function setHH(){document.documentElement.style.setProperty("--hh",document.querySelector("header").offsetHeight+"px");}addEventListener("resize",()=>{setHH();moveInd();});

/* ---------- key findings page ---------- */
function findings(){const S=D.layers.in_state.features,IN=LEVELS.in,K=D.layers.k_district.features,ker=S.find(f=>f.properties.name=="Kerala");
 const nat={};IN.forEach(l=>nat[l]=agg(S,l,false));const kin={};IN.forEach(l=>kin[l]=agg([ker],l,false));
 const types={};D.layers.in_sub.features.forEach(f=>{const t=f.properties.type;if(t&&t!="Not classified")types[t]=(types[t]||0)+1;});
 const ntypes=Object.values(types).reduce((a,b)=>a+b,0),pts=D.pts;
 $("hero").innerHTML=kpiCard("far",fmt(nat.primary.far/1e6,1),"million","primary children live more than 1 km from a primary school")
  +kpiCard("seat",fmt(nat.higher_secondary.m,2),"","higher-secondary seats per child in India: about "+fmt(nat.higher_secondary.m*10,0)+" for every 10 children")
  +kpiCard("list",fmt(ntypes,0),"","sub-districts grouped into 5 area types by the kind of problem")+kpiCard("kids",fmt(pts.length,0),"","priority areas in Kerala where children live beyond the norm");
 document.querySelectorAll("#hero .l").forEach(e=>e.style.whiteSpace="normal");
 const card=(h,sub,body,foot,wide)=>`<div class="card fcard${wide?" span2":""}"><h3>${h}</h3><p class="sub">${sub}</p>${body}${foot?`<p class="foot">${foot}</p>`:""}</div>`;
 const pct=x=>fmt(x,1)+" %",two=x=>fmt(x,2);
 let html=card("Distance hurts primary; seats are short at higher secondary","India, Government + Aided schools, by school level",
  `<div class="grp">Children beyond the distance norm</div>`+hbars(IN.map(l=>({label:`${SHORT[l]} (${NORM[l]})`,value:nat[l].b,col:"#184f95",tip:`${LV[l]}: ${pct(nat[l].b)} of children beyond ${NORM[l]}`})),{max:20,f:pct,lw:150})
  +`<div class="grp" style="margin-top:12px">Seats per child (line = 1 seat for every child)</div>`+hbars(IN.map(l=>({label:SHORT[l],value:nat[l].m,col:"#3987e5",tip:`${LV[l]}: ${two(nat[l].m)} seats per child`})),{max:1.4,ref:1,f:two,lw:150}),
  "Primary has a 1 km norm, so distance is the main gap there. At higher secondary there are only about 4 seats for every 10 children.");
 html+=card("Kerala against India, same school levels","Kerala values from the all-India run, so the levels and data match",
  `<div class="keys"><span><i style="background:#2a78d6"></i>India</span><span><i style="background:#eb6834"></i>Kerala</span></div>`
  +IN.map(l=>`<div class="grp">${LV[l]}</div>`+hbars([{label:"beyond norm",value:nat[l].b,col:"#2a78d6",tip:`India: ${pct(nat[l].b)}`},{label:"",value:kin[l].b,col:"#eb6834",tip:`Kerala: ${pct(kin[l].b)}`}],{max:20,f:pct,lw:96})
   +hbars([{label:"seats / child",value:nat[l].m,col:"#2a78d6",tip:`India: ${two(nat[l].m)}`},{label:"",value:kin[l].m,col:"#eb6834",tip:`Kerala: ${two(kin[l].m)}`}],{max:2,ref:1,f:two,lw:96})).join(""),
  "Kerala has far more seats per child at every level and is close to universal above primary, but slightly more of its primary children live beyond 1 km (hilly Idukki and Wayanad).");
 const st=S.map(f=>f.properties).filter(p=>p.c_primary>=1e5);
 const farS=st.filter(p=>!p.f_primary&&p.b_primary!=null).sort((a,b)=>b.b_primary-a.b_primary).slice(0,10);
 html+=card("States with the most primary children far from a school","Share of primary children beyond 1 km · top 10 states",
  hbars(farS.map(p=>({label:p.name,value:p.b_primary,col:"#184f95",go:p.name,tip:`<b>${p.name}</b><br>${pct(p.b_primary)} beyond 1 km · about ${fmt(p.c_primary*p.b_primary/100,0)} children`})),{f:pct,lw:150}),
  "Click a state to open it on the map. States with incomplete school lists are left out.");
 const fewS=st.filter(p=>!p.f_higher_secondary&&p.m_higher_secondary!=null).sort((a,b)=>a.m_higher_secondary-b.m_higher_secondary).slice(0,10);
 html+=card("States with the fewest higher-secondary seats","Seats per child at higher secondary (classes 11–12) · 10 lowest states",
  hbars(fewS.map(p=>({label:p.name,value:p.m_higher_secondary,col:"#256abf",go:p.name,tip:`<b>${p.name}</b><br>${two(p.m_higher_secondary)} seats per child`})),{max:1.2,ref:1,f:two,lw:150}),
  "Line = 1 seat for every child. Click a state to open it on the map.");
 const iq=(D.ineq||[]).filter(r=>r.variable=="a_main");
 html+=card("Most of the inequality is inside states","Theil index of seats per child, split into between-state and within-state parts",
  `<div class="keys"><span><i style="background:#eb6834"></i>between states</span><span><i style="background:#2a78d6"></i>within states</span></div>`
  +`<div class="hb" style="--lw:150px">`+IN.map(l=>{const r=iq.find(x=>x.level==l);if(!r)return"";const bs=r.between;
   return `<div class="hr" data-tip="${esc(`${LV[l]}: ${fmt(bs,0)} % between states, ${fmt(100-bs,0)} % within · Gini ${two(r.gini)}`)}"><span class="hl">${SHORT[l]} · Gini ${two(r.gini)}</span><span class="ht"><span class="hs"><span style="width:${bs}%;background:#eb6834"></span><span style="width:${100-bs}%;background:#2a78d6"></span></span></span><span class="hv">${fmt(bs,0)} %</span></div>`;}).join("")+`</div>`,
  "The number on the right is the between-state share. At primary only 9 % lies between states, so planning by district and block matters most.");
 html+=card("Five kinds of problem areas","K-Means groups of India sub-districts (silhouette 0.42)",
  hbars(Object.keys(TYPEC).filter(k=>types[k]).sort((a,b)=>types[b]-types[a]).map(k=>({label:k,value:types[k],col:TYPEC[k],tip:`<b>${k}</b><br>${TYPED[k]}`})),{f:x=>fmt(x,0),lw:200}),
  "Hover a bar for what the group means. The crowded group is the biggest: schools are near, but seats are short.");
 const kd=K.map(f=>f.properties).filter(p=>p.b_lp!=null).sort((a,b)=>b.b_lp-a.b_lp);
 html+=card("Kerala: lower primary children beyond 1 km, by district","The main gap in Kerala is distance for its youngest pupils",
  hbars(kd.map(p=>({label:p.name,value:p.b_lp,col:"#184f95",tip:`<b>${p.name}</b><br>${pct(p.b_lp)} of LP children beyond 1 km · ${two(p.m_lp)} seats per child`})),{f:pct,lw:150}),
  "Hilly districts (Idukki, Wayanad) are far above the rest.");
 const byL={};pts.forEach(p=>{byL[p.level]=byL[p.level]||{n:0,c:0};byL[p.level].n++;byL[p.level].c+=p.children;});
 const big5=pts.filter(p=>p.level=="lp").sort((a,b)=>b.children-a.children).slice(0,5);
 html+=card("Kerala priority areas for new schools","Connected 100 m cells beyond the norm with at least 50 children",
  `<div class="grp">Areas by level</div>`+hbars(LEVELS.k.filter(l=>byL[l]).map(l=>({label:`${SHORT[l]} · ${fmt(byL[l].c,0)} children`,value:byL[l].n,col:"#e87ba4",tip:`${LV[l]}: ${byL[l].n} areas, ${fmt(byL[l].c,0)} children`})),{f:x=>fmt(x,0),lw:190})
  +`<div class="grp" style="margin-top:12px">Largest lower-primary areas (children)</div>`+hbars(big5.map(p=>({label:`${p.local_body}, ${p.district}`,value:p.children,col:"#e87ba4",tip:`${p.cells} cells of 100 m`})),{f:x=>fmt(x,0),lw:190}),
  "They show where the gap is, not the exact site: choosing a site needs land and cost data.");
 $("fgrid").innerHTML=html;
 $("fgrid").addEventListener("click",e=>{const r=e.target.closest("[data-go]");if(!r)return;goState(r.dataset.go);});
 $("fgrid").addEventListener("keydown",e=>{const r=e.target.closest("[data-go]");if(r&&(e.key=="Enter"||e.key==" ")){e.preventDefault();goState(r.dataset.go);}});}
function goState(name){page("app");if($("view").value!="in"){$("view").value="in";setView();}$("state").value=name;setState();}

/* ---------- about page ---------- */
function about(){$("aboutw").innerHTML=`<h2 class="ttl">About the data</h2><p class="lead">How the numbers on this site were made, in plain words.</p>
<div class="fgrid"><div class="card fcard"><h3>How it was calculated</h3><ol>
<li><b>Children (demand):</b> WorldPop 2026 population in 100 m cells, split into school ages, times the share of pupils in Government + Aided schools (UDISE).</li>
<li><b>Seats (supply):</b> teachers × the official pupils-per-teacher norm; higher secondary from the real HSCAP and VHSCAP batches. A check run uses the pupils enrolled now.</li>
<li><b>Catchment:</b> a child may travel 1 km (primary), 3 km (upper primary), 5 km (secondary; 3 km in hilly Kerala) or 7 km (higher secondary).</li>
<li><b>Nearer counts more:</b> a Gaussian weight, 1 at the school and 0 at the edge of the catchment.</li>
<li><b>E2SFCA:</b> each school's seats are shared among the children around it; each cell adds up what it can reach. That is “seats per child”.</li>
<li><b>Areas:</b> cells are added up into states, districts, sub-districts, taluks and local bodies, weighted by children.</li>
<li><b>Patterns:</b> Gi* hot spots, K-Means area types, FP-Growth rules and Kerala priority areas.</li></ol>
<p class="foot">Computed with Apache Spark and Apache Sedona over 140 million populated cells and about 1.07 million schools.</p></div>
<div class="card fcard"><h3>Data notes</h3><div id="notes"></div></div></div>`;
 $("notes").innerHTML=D.panels.Notes;}

/* ---------- page switching, theme, tables ---------- */
function moveInd(){const b=document.querySelector("#pages button.on"),i=$("ind");if(!b||!i)return;i.style.left=b.offsetLeft+"px";i.style.width=b.offsetWidth+"px";}
function page(p){document.querySelectorAll(".view").forEach(v=>v.classList.toggle("on",v.id==p));document.querySelectorAll("#pages button").forEach(b=>{const on=b.dataset.p==p;b.classList.toggle("on",on);if(on)b.setAttribute("aria-current","page");else b.removeAttribute("aria-current");});moveInd();if(p=="app")setTimeout(()=>map.invalidateSize(),0);else document.getElementById(p).scrollTo({top:0});}
$("pages").onclick=e=>{const p=e.target.dataset.p;if(p)page(p);};addEventListener("resize",moveInd);
function isDark(){const t=document.documentElement.getAttribute("data-theme");return t?t=="dark":matchMedia("(prefers-color-scheme: dark)").matches;}
$("theme").onclick=()=>{const t=isDark()?"light":"dark";document.documentElement.setAttribute("data-theme",t);try{localStorage.setItem("sae-theme",t);}catch(e){}draw(false);};
const tabs=Object.keys(D.panels).filter(t=>t!="Notes");$("tabs").innerHTML=tabs.map((t,i)=>`<button role="tab" aria-selected="${!i}" data-t="${t}" class="${i?"":"on"}">${t}</button>`).join("");
$("panel").innerHTML=D.panels[tabs[0]];$("tabs").onclick=e=>{const t=e.target.dataset.t;if(!t)return;document.querySelectorAll("#tabs button").forEach(b=>{b.classList.toggle("on",b.dataset.t==t);b.setAttribute("aria-selected",b.dataset.t==t);});$("panel").innerHTML=D.panels[t];};
let INIT=true;const H0=location.hash;  // read the shared link before the first draw rewrites it
setHH();setView();applyHash(H0);findings();about();moveInd();
INIT=false;{const ld=$("loading");if(ld){ld.style.opacity="0";setTimeout(()=>ld.remove(),RM?0:220);}}
</script></body></html>"""
INEQ = ineq_in[ineq_in["scope"].eq("India")].rename(columns={"between_share_%": "between"})[["level", "variable", "gini", "between"]]
ineq_rows = [{k: (None if pd.isna(v) else (round(float(v), 4) if isinstance(v, float) else v)) for k, v in r.items()} for r in INEQ.to_dict("records")]
assert len(ineq_rows) == 12, len(ineq_rows)  # 4 levels x 3 variables
DATA = {"lv": LV, "layers": LAYERS, "gaps": GAPS, "pts": pts, "panels": PANELS, "ineq": ineq_rows}
data_json = json.dumps(DATA, separators=(",", ":"), ensure_ascii=False)
assert "</script" not in data_json
html = TEMPLATE.replace("__LEAFLET_CSS__", LEAFLET_CSS).replace("__LEAFLET_JS__", LEAFLET_JS).replace("__DATA__", data_json)
OUT_FILE = OUT / "school_access_dashboard.html"
OUT_FILE.write_text(html, encoding="utf-8")
print(f"written {OUT_FILE.relative_to(PROJECT)}: {OUT_FILE.stat().st_size / 1e6:.1f} MB")

written FINAL COMPLETE DATASET\results\dashboard\school_access_dashboard.html: 12.7 MB


## Checks

In [9]:
# Checks: the embedded data parses, every layer is complete, and 10 random units per layer equal the CSVs
text = OUT_FILE.read_text(encoding="utf-8")
start = text.rindex("const D=") + len("const D=")  # the data comes after the inlined Leaflet code
back = json.loads(text[start:text.index(";\nconst LV=D.lv")])
counts = {k: len(v["features"]) for k, v in back["layers"].items()}
assert counts == {"in_state": len(by_state), "in_district": len(by_district), "in_sub": len(by_sub),
                  "k_district": len(k_district), "k_taluk": len(k_taluk), "k_lb": len(k_lb)}, counts
src = {"in_state": (by_state, "state", "name", LEVELS), "in_district": (by_district, "district", "name", LEVELS),
       "k_lb": (k_lb, "local_body", "name", LEVELS_K), "k_taluk": (k_taluk, "subdistrict", "name", LEVELS_K)}
random.seed(1)
n = 0
for key, (t, col, prop, lvs) in src.items():
    t = t.drop_duplicates(col, keep=False).set_index(col)  # names that are unique, to look rows up by name
    feats = [f for f in back["layers"][key]["features"] if f["properties"]["name"] in t.index]
    for f in random.sample(feats, min(10, len(feats))):
        row = t.loc[f["properties"]["name"]]
        for lv in lvs:
            for short, col_, nd in [("b", "beyond_norm_%", 1), ("n", "mean_nearest_km", 2), ("m", "a_main", 3), ("e", "a_enrol", 3)]:
                got, want = f["properties"][f"{short}_{lv}"], row[f"{col_}_{lv}"]
                assert (got is None and pd.isna(want)) or abs(got - want) <= 0.5 * 10 ** -nd + 1e-9, (key, f["properties"]["name"], short, lv)
        n += 1
# state picker: every district / sub-district names one of the 36 states, and every state has its own districts
st_names = {f["properties"]["name"] for f in back["layers"]["in_state"]["features"]}
for key in ["in_district", "in_sub"]:
    per = pd.Series([f["properties"]["state"] for f in back["layers"][key]["features"]]).value_counts()
    assert set(per.index) <= st_names, (key, set(per.index) - st_names)
    if key == "in_district":
        assert set(per.index) == st_names, st_names - set(per.index)
        print("districts per state/UT: min", per.min(), "max", per.max(), "| states", len(per))
# holes: one grey patch each, values equal the cells inside
assert len(back["gaps"]["features"]) == len(holes)
for f, i in zip(back["gaps"]["features"], holes.index):
    h = hv.loc[i]
    assert f["properties"]["c_lp"] == round(h["children_lp"]) and f["properties"]["b_lp"] == round(h["beyond_norm_%_lp"], 2)
assert OUT_FILE.stat().st_size < 20e6, "dashboard file over 20 MB"
print(f"layers complete {counts}; {n} random units equal the CSVs (all levels, 4 measures); file {OUT_FILE.stat().st_size / 1e6:.1f} MB")

districts per state/UT: min 1 max 75 | states 36
layers complete {'in_state': 36, 'in_district': 735, 'in_sub': 6810, 'k_district': 14, 'k_taluk': 77, 'k_lb': 1027}; 40 random units equal the CSVs (all levels, 4 measures); file 12.7 MB
